# A Charging-Data Digital Twin for IoT-Connected Lithium-Ion Batteries

Code for the paper *"A Charging-Data Digital Twin for IoT-Connected Lithium-Ion Batteries: Health, Charge and Remaining-Life Estimation with Calibrated Uncertainty and Tolerance to Sensor Faults"*.

Every number, table and figure of the paper is produced by this notebook.

| Part | What it does |
|---|---|
| A | NASA cells: charging features, end-of-life rule, leave-one-battery-out comparison of five models |
| B | NASA cells: self-updating twin for RUL with 90% conformal bands |
| C | LFP cells (MIT–Stanford–Toyota): download, feature extraction, cleaning |
| D | LFP cells: grouped 5-fold comparison of five models |
| E | LFP cells: self-updating twin and confidence bands |
| F | NASA cells: SOC estimation with SOH-corrected charge counting |
| G | Monitoring dashboard (replay of the digital twin) and its pictures |
| H | Final tables of the paper |
| I | Model cost, sensor-fault study, statistical tests, input importance, fault-tolerance remedies |
| J | Paper figures |
| K | Additional analyses: data size per charge, software and hardware, effect sizes, unseen charging policies |
| L | NASA cells: single fixed split (LSTM vs CNN-LSTM on B0018) and charge and energy balance checks |

**How to run**
1. Open the notebook in Google Colab and choose *Runtime → Change runtime type → T4 GPU*.
2. Put the NASA battery data set (files `B0005.mat`, `B0006.mat`, `B0007.mat`, `B0018.mat`) anywhere inside your Google Drive. The LFP data set is downloaded automatically in Part C.
3. *Runtime → Run all* and allow access to Drive when asked.

Outside Colab, set the environment variable `DATA_ROOT` to the folder that holds the NASA files (default: `./data`).

Slow steps save their results in a cache folder (`FYP_cache`) and load them on the next run. Set `RERUN_ALL = True` in the setup cells to recompute everything. A full run from scratch takes a few hours on a T4 GPU.

# Part A — NASA dataset

In [ ]:
LOCAL_ROOT = "data"         # folder used outside Colab when DATA_ROOT is not set (ignored in Colab)
import os, glob, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)

try:                                    # in Colab: mount Drive
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive"
except ImportError:                     # outside Colab: set DATA_ROOT yourself
    ROOT = os.environ.get("DATA_ROOT", LOCAL_ROOT)
    os.makedirs(ROOT, exist_ok=True)

hits = glob.glob(os.path.join(ROOT, "**", "B0005.mat"), recursive=True)
if not hits:
    raise FileNotFoundError(f"B0005.mat not found under {ROOT}. Put the NASA battery data set (B0005.mat ... B0018.mat) inside it.")
battery_path = os.path.dirname(hits[0])

OUT = os.path.join(ROOT, "FYP_NASA_Battery", "stage2_results")
os.makedirs(OUT, exist_ok=True)

BATTERIES = ["B0005", "B0006", "B0007", "B0018"]
RATED_CAPACITY_AH = 2.0                 # nominal capacity of the NASA 18650 cells
print("Dataset folder:", battery_path)
print("Results folder:", OUT)

# ---- saved results: a finished heavy step is stored in the cache folder and loaded next time instead of recomputed ----
import pickle
RERUN_ALL = False                      # set to True to ignore saved results and recompute everything
CACHE_DIR = os.path.join(ROOT, "FYP_cache"); os.makedirs(CACHE_DIR, exist_ok=True)
def have(name): return (not RERUN_ALL) and os.path.exists(os.path.join(CACHE_DIR, name + ".pkl"))
def save(name, obj):
    with open(os.path.join(CACHE_DIR, name + ".pkl"), "wb") as fh: pickle.dump(obj, fh)
def load(name):
    print(f"Loaded saved results '{name}' (set RERUN_ALL = True in the setup cell to recompute)")
    with open(os.path.join(CACHE_DIR, name + ".pkl"), "rb") as fh: return pickle.load(fh)

## A2. Cycle-level dataset with charging features
Each row is one discharge cycle. Its capacity is the label. Its inputs come from two places:

- **Discharge features** — eight summary values of each discharge, kept only for comparison.
- **Charging features** (new) — taken from the charge that came just before that discharge:

| Feature | Meaning |
|---|---|
| `CC_time_s` | Time spent in the constant-current phase (until 4.2 V) |
| `T_40_42_s` | Time for the voltage to rise from 4.0 V to 4.2 V (a *partial* charge window) |
| `Q_cc_Ah` | Charge delivered during the constant-current phase |
| `CV_time_s` | Time spent in the constant-voltage phase |

Charging temperature is still extracted but **not used as an input**: in version 1 it did not follow ageing consistently across cells.
A charge whose constant-current time jumps more than 20% away from its neighbours (an interrupted or partial charge) is replaced by interpolation.

In [ ]:
V_LOW, V_HIGH = 4.0, 4.195              # partial-charge voltage window
I_CV_END = 0.10                         # A: current at which we treat the CV phase as finished

def arr(d, key):
    return np.asarray(d[key]).flatten().astype(float)

def charge_features(d):
    """Health features from one charge cycle. Returns NaNs if the charge is incomplete/abnormal."""
    nan = dict(CC_time_s=np.nan, T_40_42_s=np.nan, Q_cc_Ah=np.nan,
               CV_time_s=np.nan, Chg_Tmax=np.nan, Chg_dT=np.nan)
    t, v, i, temp = arr(d, "Time"), arr(d, "Voltage_measured"), arr(d, "Current_measured"), arr(d, "Temperature_measured")
    if len(t) < 50 or t[-1] < 1000:                      # too short to be a real charge
        return nan
    t, v, i, temp = t[2:], v[2:], i[2:], temp[2:]        # the first samples are switching glitches
    if v[0] >= V_LOW or v.max() < V_HIGH:                # must pass through the whole window
        return nan
    i_low = int(np.argmax(v >= V_LOW))
    i_high = int(np.argmax(v >= V_HIGH))
    if i_high <= i_low:
        return nan
    after = np.where(i[i_high:] < I_CV_END)[0]
    i_end = i_high + int(after[0]) if len(after) else len(t) - 1
    i_pos = np.clip(i[:i_high + 1], 0, None)
    q_cc = np.sum((i_pos[:-1] + i_pos[1:]) / 2 * np.diff(t[:i_high + 1])) / 3600
    return dict(CC_time_s=t[i_high] - t[0], T_40_42_s=t[i_high] - t[i_low], Q_cc_Ah=q_cc,
                CV_time_s=t[i_end] - t[i_high], Chg_Tmax=temp.max(), Chg_dT=temp.max() - temp[0])

def load_battery(name):
    cycles = loadmat(os.path.join(battery_path, name + ".mat"))[name][0, 0]["cycle"]
    rows, last_charge, k = [], None, 0
    for idx in range(cycles.shape[1]):
        c = cycles[0, idx]
        ctype = str(c["type"][0])
        if ctype == "charge":
            last_charge = charge_features(c["data"][0, 0])
        elif ctype == "discharge":
            d = c["data"][0, 0]
            cap = arr(d, "Capacity")
            if len(cap) == 0 or not np.isfinite(cap[0]):
                continue
            k += 1
            v, i, temp, t = arr(d, "Voltage_measured"), arr(d, "Current_measured"), arr(d, "Temperature_measured"), arr(d, "Time")
            row = dict(Battery=name, Cycle=idx + 1, k=k, Capacity_Ah=cap[0],
                       Voltage_mean=v.mean(), Voltage_min=v.min(), Voltage_max=v.max(),
                       Current_mean=i.mean(), Temperature_mean=temp.mean(), Temperature_max=temp.max(),
                       Discharge_time_s=t[-1] - t[0])
            row.update(last_charge if last_charge is not None else charge_features({"Time": [0], "Voltage_measured": [0], "Current_measured": [0], "Temperature_measured": [0]}))
            rows.append(row)
            last_charge = None                           # a charge is used for one discharge only
    return pd.DataFrame(rows)

CHARGE_FEATURES = ["CC_time_s", "T_40_42_s", "Q_cc_Ah", "CV_time_s"]
ALL_CHARGE_COLS = CHARGE_FEATURES + ["Chg_Tmax", "Chg_dT"]
DISCHARGE_FEATURES = ["Cycle", "Voltage_mean", "Voltage_min", "Voltage_max", "Current_mean",
                   "Temperature_mean", "Temperature_max", "Discharge_time_s"]

frames = []
for name in BATTERIES:
    b = load_battery(name)
    med = b["CC_time_s"].rolling(7, center=True, min_periods=1).median()
    bad = b["CC_time_s"].isna() | ((b["CC_time_s"] - med).abs() > 0.20 * med)      # missing or partial charge
    b.loc[bad, ALL_CHARGE_COLS] = np.nan
    b["Charge_feature_filled"] = bad
    b[ALL_CHARGE_COLS] = b[ALL_CHARGE_COLS].interpolate(limit_direction="both")     # fill from neighbouring cycles
    print(f"{name}: {len(b)} discharge cycles, {int(bad.sum())} with an abnormal/missing charge (filled from neighbouring cycles)")
    frames.append(b)
df = pd.concat(frames, ignore_index=True)
df.head()

## A3. One end-of-life definition for every cell
- **SOH** = capacity / rated capacity (2.0 Ah).
- **End of life** = first cycle where capacity falls to the threshold. It starts at 1.4 Ah and is raised in 0.05 Ah steps until every cell reaches it.
- **RUL** = discharge cycles left until end of life, scored only on cycles before end of life.

In [ ]:
EOL_CAPACITY_AH = 1.40

def first_eol(b, thr):
    hit = b.index[b["Capacity_Ah"] <= thr]
    return int(b.loc[hit[0], "k"]) if len(hit) else None

thr = EOL_CAPACITY_AH
while any(first_eol(b, thr) is None for _, b in df.groupby("Battery")):
    thr = round(thr + 0.05, 2)
EOL_USED_AH = thr
SOH_EOL = EOL_USED_AH / RATED_CAPACITY_AH

df["SOH"] = df["Capacity_Ah"] / RATED_CAPACITY_AH
df["EOL_k"] = df["Battery"].map({n: first_eol(b, EOL_USED_AH) for n, b in df.groupby("Battery")})
df["RUL"] = (df["EOL_k"] - df["k"]).clip(lower=0)
df["Before_EOL"] = df["k"] <= df["EOL_k"]
print(f"EOL threshold used: {EOL_USED_AH} Ah  (SOH = {SOH_EOL:.1%})")
df.groupby("Battery").agg(Discharge_cycles=("k", "max"), EOL_cycle=("EOL_k", "first"))

## A4. Do the charging features track battery health?

In [ ]:
corr = pd.DataFrame({n: b[CHARGE_FEATURES + ["Discharge_time_s"]].corrwith(b["Capacity_Ah"])
                     for n, b in df.groupby("Battery")})
print(corr)
corr.to_csv(os.path.join(OUT, "feature_capacity_correlation.csv"))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for name, b in df.groupby("Battery"):
    axes[0].plot(b["k"], b["Capacity_Ah"], label=name)
    axes[1].plot(b["k"], b["CC_time_s"], label=name)
    axes[2].plot(b["k"], b["T_40_42_s"], label=name)
axes[0].axhline(EOL_USED_AH, color="black", ls="--", lw=1)
for ax, yl in zip(axes, ["Capacity (Ah)", "CC charging time (s)", "4.0 V to 4.2 V charging time (s)"]):
    ax.set_xlabel("Discharge cycle"); ax.set_ylabel(yl); ax.grid(True, alpha=0.3)
axes[0].legend(frameon=False); plt.tight_layout()
plt.savefig(os.path.join(OUT, "charging_features_vs_cycle.png"), dpi=300); plt.show()

## A5. Deep model definitions (LSTM and hybrid CNN-LSTM)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEQ_LEN = 20
N_SEEDS = 5          # training runs averaged for each deep model
EPOCHS = 100

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

def build_model(n_features, kind="CNN-LSTM"):
    inp = keras.Input((SEQ_LEN, n_features))
    x = inp
    if kind == "CNN-LSTM":
        x = layers.Conv1D(64, 3, padding="causal", activation="relu")(x)
        x = layers.Conv1D(64, 3, padding="causal", activation="relu")(x)
    x = layers.LSTM(64)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation="relu")(x)
    m = keras.Model(inp, layers.Dense(2)(x))
    m.compile(optimizer="adam", loss="mse")
    return m

def fit_model(m, X, y, Xv, yv):
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    m.fit(X, y, validation_data=(Xv, yv), epochs=EPOCHS, batch_size=16, callbacks=[stop], verbose=0)
    return m

## A6. Leave-one-battery-out comparison
Two input options (discharge features and charging features), each tried with five models:

| Model | What it sees |
|---|---|
| Linear (ridge) | The latest cycle only. The sanity check. |
| Random Forest | Window summary: latest value, window mean and window trend of every feature |
| Tuned XGBoost | Same window summary. Hyperparameters are searched with cross-validation **grouped by battery**, so tuning never sees the test cell |
| LSTM | The full 20-cycle window |
| Hybrid CNN-LSTM | The full 20-cycle window |

If a simple model matches the deep ones, the deep model is not adding value for that target, and the paper should say so.

Takes roughly 15 to 20 minutes on Colab.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from xgboost import XGBRegressor

XGB_SEARCH = dict(n_estimators=[100, 200, 400, 600], max_depth=[2, 3, 4, 6], learning_rate=[0.01, 0.03, 0.05, 0.1],
                  subsample=[0.6, 0.8, 1.0], colsample_bytree=[0.6, 0.8, 1.0], min_child_weight=[1, 3, 5],
                  reg_lambda=[0.5, 1.0, 5.0])
XGB_TRIALS = 15
BEST_XGB = []                                                  # chosen hyperparameters, saved for the paper

def window_summary(X):
    # (cycles, window, features) array -> table of latest value, window mean and window trend
    return np.concatenate([X[:, -1, :], X.mean(axis=1), X[:, -1, :] - X[:, 0, :]], axis=1)

CONFIGS = {
    "Discharge features":              dict(feats=DISCHARGE_FEATURES,      soh="SOH"),
    "Charging":                        dict(feats=CHARGE_FEATURES,      soh="SOH"),
}
MODELS = ["Linear", "Random Forest", "Tuned XGBoost", "LSTM", "CNN-LSTM"]

def make_sequences(b, feats, soh_col, xs, ys):
    X = xs.transform(b[feats].values)
    y = ys.transform(b[[soh_col, "RUL"]].values)
    idx = range(SEQ_LEN - 1, len(b))
    return (np.array([X[i - SEQ_LEN + 1:i + 1] for i in idx]), np.array([y[i] for i in idx]),
            b.iloc[SEQ_LEN - 1:].reset_index(drop=True))

def metrics(a, p):
    return dict(RMSE=float(np.sqrt(mean_squared_error(a, p))), MAE=float(mean_absolute_error(a, p)), R2=float(r2_score(a, p)))

def run_fold(test_name, feats, soh_col, kind):
    train = df[df["Battery"] != test_name]
    test = df[df["Battery"] == test_name].reset_index(drop=True)
    xs = StandardScaler().fit(train[feats].values)             # scalers see the training cells only
    ys = StandardScaler().fit(train[[soh_col, "RUL"]].values)
    parts = [make_sequences(b.reset_index(drop=True), feats, soh_col, xs, ys) for _, b in train.groupby("Battery")]
    X = np.concatenate([p[0] for p in parts]); y = np.concatenate([p[1] for p in parts])
    groups = np.concatenate([np.full(len(p[0]), g) for g, p in enumerate(parts)])   # which training cell each row is from
    Xt, yt, meta = make_sequences(test, feats, soh_col, xs, ys)
    preds = []
    if kind == "Linear":
        preds.append(ys.inverse_transform(Ridge(alpha=1.0).fit(X[:, -1, :], y).predict(Xt[:, -1, :])))
    elif kind == "Random Forest":
        rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=0, n_jobs=-1)
        preds.append(ys.inverse_transform(rf.fit(window_summary(X), y).predict(window_summary(Xt))))
    elif kind == "Tuned XGBoost":
        W, Wt, out = window_summary(X), window_summary(Xt), []
        for j, target in enumerate(["SOH", "RUL"]):            # one tuned model per target
            search = RandomizedSearchCV(XGBRegressor(objective="reg:squarederror", random_state=0, n_jobs=1),
                                        XGB_SEARCH, n_iter=XGB_TRIALS, cv=GroupKFold(n_splits=len(parts)),
                                        scoring="neg_root_mean_squared_error", random_state=0, n_jobs=-1)
            search.fit(W, y[:, j], groups=groups)
            BEST_XGB.append(dict(Test_cell=test_name, Target=target, N_inputs=len(feats), **search.best_params_))
            out.append(search.predict(Wt))
        preds.append(ys.inverse_transform(np.column_stack(out)))
    else:
        for seed in range(N_SEEDS):
            set_seed(seed)
            order = np.random.permutation(len(X)); n_val = max(1, int(0.15 * len(X)))
            va, tr = order[:n_val], order[n_val:]
            m = fit_model(build_model(len(feats), kind), X[tr], y[tr], X[va], y[va])
            preds.append(ys.inverse_transform(m.predict(Xt, verbose=0)))
    preds = np.array(preds)                                    # (runs, cycles, 2)
    if soh_col == "SOH_rel":                                   # back to the common SOH scale (rated capacity)
        preds[:, :, 0] *= meta["Cap0"].values / RATED_CAPACITY_AH
    return meta, preds

if have('A6_v3'):
    RESULTS, PRED, BEST_XGB = load('A6_v3')
else:
    RESULTS, PRED = [], {}
    for cfg_name, cfg in CONFIGS.items():
        for kind in MODELS:
            for test_name in BATTERIES:
                meta, preds = run_fold(test_name, cfg["feats"], cfg["soh"], kind)
                mean = preds.mean(axis=0)
                mean[:, 1] = np.clip(mean[:, 1], 0, None)
                PRED[(cfg_name, kind, test_name)] = dict(meta=meta, preds=preds, mean=mean)
                pre = meta["Before_EOL"].values                     # RUL is scored before EOL only
                soh, rul = metrics(meta["SOH"].values, mean[:, 0]), metrics(meta["RUL"].values[pre], mean[pre, 1])
                RESULTS.append(dict(Inputs=cfg_name, Model=kind, Test_cell=test_name,
                                    SOH_RMSE=soh["RMSE"], SOH_MAE=soh["MAE"], SOH_R2=soh["R2"],
                                    RUL_RMSE=rul["RMSE"], RUL_MAE=rul["MAE"], RUL_R2=rul["R2"]))
            print("done:", cfg_name, "|", kind)
    save('A6_v3', (RESULTS, PRED, BEST_XGB))
results = pd.DataFrame(RESULTS)
results.to_csv(os.path.join(OUT, "lobo_results_per_cell.csv"), index=False)
average = results.groupby(["Inputs", "Model"], sort=False).mean(numeric_only=True)
average.to_csv(os.path.join(OUT, "lobo_results_average.csv"))
pd.DataFrame(BEST_XGB).to_csv(os.path.join(OUT, "xgboost_best_hyperparameters.csv"), index=False)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
print("\nSOH RMSE per test cell");  print(results.pivot_table(index=["Inputs", "Model"], columns="Test_cell", values="SOH_RMSE", sort=False))
print("\nRUL RMSE per test cell (cycles)"); print(results.pivot_table(index=["Inputs", "Model"], columns="Test_cell", values="RUL_RMSE", sort=False))
print("\nAverage over the four test cells"); print(average)

## A7. Figures: actual vs predicted on every test cell (charging features)

In [ ]:
PLOT = {("Charging", "Tuned XGBoost"): dict(color="tab:green", ls=":"),
        ("Charging", "LSTM"): dict(color="tab:blue", ls="--"),
        ("Charging", "CNN-LSTM"): dict(color="tab:brown", ls="-")}
for j, (target, ylabel, scale) in enumerate([("SOH", "SOH (%)", 100), ("RUL", "RUL (cycles)", 1)]):
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.3), sharey=False)
    for ax, test_name in zip(axes, BATTERIES):
        meta = PRED[(*list(PLOT)[0], test_name)]["meta"]
        ax.plot(meta["k"], meta[target] * scale, color="black", lw=1.8, label="Actual")
        for key, style in PLOT.items():
            p = PRED[(*key, test_name)]
            ax.plot(meta["k"], p["mean"][:, j] * scale, lw=1.4, label=f"{key[0]} ({key[1]})", **style)
            sd = p["preds"][:, :, j].std(axis=0) * scale
            ax.fill_between(meta["k"], p["mean"][:, j] * scale - 2 * sd, p["mean"][:, j] * scale + 2 * sd,
                            color=style["color"], alpha=0.15)
        ax.set_title(f"Test cell {test_name}", fontsize=10); ax.set_xlabel("Discharge cycle"); ax.grid(True, alpha=0.3)
    axes[0].set_ylabel(ylabel); axes[0].legend(frameon=False, fontsize=8); plt.tight_layout()
    plt.savefig(os.path.join(OUT, f"LOBO_{target}.png"), dpi=300); plt.show()
print("Saved figures and tables to", OUT)

# Part B — Self-updating twin for RUL (experimental)
1. **SOH virtual sensor:** a linear model turns the charging features into an SOH estimate every cycle.
2. **Self-updating trend:** as cycles stream in, the twin fits the battery's own ageing trend and projects it to end of life. It starts from the average ageing rate of the training batteries and trusts the battery's own data more as it arrives.
3. **Adaptive Tuned XGBoost twin:** the frozen Tuned XGBoost prediction and the self-updating prediction are fused, each weighted by how certain it is.
4. **Calibrated 90% bands:** the twin is replayed on each training battery as if it were new, and the band is widened until it would have covered 90% of those cases.

**Honest status.** In the first version the trend overshot early in life and the twin did not beat the frozen models. This version adds two safeguards (see B1). Whether they are enough is what this run shows.

## B1. Building blocks
Safeguards against early overshoot:
- **Noise level:** the trend now treats the natural bumps in a battery's ageing curve as noise, so a few flat cycles no longer convince it that ageing has stopped.
- **Rate floor:** the projected ageing rate is never allowed below `RATE_FLOOR` times the slowest training battery.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, GroupKFold, cross_val_predict
from xgboost import XGBRegressor

FEATS = CHARGE_FEATURES
FORGET = 0.98        # weight of a cycle shrinks by 2% for every cycle it lies in the past
N_DRAWS = 400        # number of plausible trend lines drawn at each cycle
RUL_CAP = 300        # upper limit for a projected RUL (cycles)
RATE_FLOOR = 0.5     # ageing is assumed at least this fraction of the slowest training battery's rate
Z90 = 1.645          # 90% band for a normal distribution
XGB_SEARCH = dict(n_estimators=[100, 200, 400, 600], max_depth=[2, 3, 4, 6], learning_rate=[0.01, 0.03, 0.05, 0.1],
                  subsample=[0.6, 0.8, 1.0], colsample_bytree=[0.6, 0.8, 1.0], min_child_weight=[1, 3, 5],
                  reg_lambda=[0.5, 1.0, 5.0])
XGB_TRIALS = 15

def windows(b, xs):
    # rows from the 20th cycle on: the raw 20-cycle window and its summary table (latest, mean, trend)
    X = xs.transform(b[FEATS].values)
    seq = np.array([X[i - SEQ_LEN + 1:i + 1] for i in range(SEQ_LEN - 1, len(b))])
    return seq, np.concatenate([seq[:, -1], seq.mean(axis=1), seq[:, -1] - seq[:, 0]], axis=1)

def online_rul(k, soh_est, prior, sigma, sigma_bias, rng, slowest):
    # Streaming RUL: at each cycle use only the SOH estimates seen so far.
    # Returns per cycle: median, 5th, 95th, 16th, 84th percentile of the projected RUL.
    intercept0, slope0, slope_sd = prior
    out = []
    for i in range(len(k)):
        x = (k[:i + 1] - k[i]).astype(float)                     # 0 for the current cycle, negative for the past
        w = FORGET ** (-x)
        A = np.column_stack([np.ones(i + 1), x])                 # SOH = level_now + rate * x
        m0 = np.array([intercept0 + slope0 * k[i], slope0])      # what the training batteries suggest
        P0_inv = np.diag([1 / 0.10 ** 2, 1 / slope_sd ** 2])
        Aw = A * w[:, None]
        cov = np.linalg.inv(P0_inv + Aw.T @ A / sigma ** 2)
        mean = cov @ (P0_inv @ m0 + Aw.T @ soh_est[:i + 1] / sigma ** 2)
        draws = rng.multivariate_normal(mean, cov, N_DRAWS)
        level = draws[:, 0] + rng.normal(0, sigma_bias, N_DRAWS)  # allow for a fixed offset of the SOH sensor
        rate = np.minimum(draws[:, 1], -RATE_FLOOR * slowest)      # never slower than the floor
        rul = (SOH_EOL - level) / rate
        out.append(np.percentile(np.clip(rul, 0, RUL_CAP), [50, 5, 95, 16, 84]))
    return np.array(out)

def band(pred, sd, z=Z90):
    return np.clip(pred - z * sd, 0, None), pred + z * sd

def fuse(online, online_sd, frozen, frozen_sd):
    # inverse-variance weighting: the more certain prediction counts for more
    w_on, w_fr = 1 / online_sd ** 2, 1 / frozen_sd ** 2
    return (w_on * online + w_fr * frozen) / (w_on + w_fr), np.sqrt(1 / (w_on + w_fr)), w_on / (w_on + w_fr)

## B2. Leave-one-battery-out run
For each test battery: train on the others, then stream the test battery cycle by cycle. The CNN-LSTM predictions are reused from Part A.

In [ ]:
def run_twin(test_name):
    train = df[df["Battery"] != test_name].reset_index(drop=True)
    test = df[df["Battery"] == test_name].reset_index(drop=True)
    cells = [b.reset_index(drop=True) for _, b in train.groupby("Battery")]
    xs = StandardScaler().fit(train[FEATS].values)

    # ---- 1. SOH virtual sensor (its error is measured on training cells it did not see) ----
    Xtr, g = xs.transform(train[FEATS].values), train["Battery"].values
    oof = cross_val_predict(Ridge(alpha=1.0), Xtr, train["SOH"].values, cv=GroupKFold(n_splits=len(cells)), groups=g)
    res = pd.Series(train["SOH"].values - oof)
    offsets = res.groupby(g).transform("mean")
    sigma, sigma_bias = float((res - offsets).std()), float(np.sqrt((offsets ** 2).mean()))
    soh_est = Ridge(alpha=1.0).fit(Xtr, train["SOH"].values).predict(xs.transform(test[FEATS].values))

    # ---- what the training batteries say about ageing rate (the prior) ----
    fits = [np.polyfit(b.loc[b["Before_EOL"], "k"], b.loc[b["Before_EOL"], "SOH"], 1) for b in cells]
    slopes, intercepts = np.array([f[0] for f in fits]), np.array([f[1] for f in fits])
    bumps = np.concatenate([b.loc[b["Before_EOL"], "SOH"] - np.polyval(f, b.loc[b["Before_EOL"], "k"]) for b, f in zip(cells, fits)])
    sigma = float(np.sqrt(sigma ** 2 + bumps.std() ** 2))        # sensor noise plus natural bumps around a straight trend
    prior = (intercepts.mean(), slopes.mean(), max(slopes.std(ddof=1), 0.25 * abs(slopes.mean())))

    # ---- 2. frozen models (trained once, never updated) ----
    parts = [windows(b, xs) for b in cells]
    S = np.concatenate([p[0] for p in parts]); W = np.concatenate([p[1] for p in parts])
    y_rul = np.concatenate([b["RUL"].values[SEQ_LEN - 1:] for b in cells])
    y_soh = np.concatenate([b["SOH"].values[SEQ_LEN - 1:] for b in cells])
    groups = np.concatenate([np.full(len(p[0]), i) for i, p in enumerate(parts)])
    St, Wt = windows(test, xs)
    meta = test.iloc[SEQ_LEN - 1:].reset_index(drop=True)
    P = {}

    rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=0, n_jobs=-1).fit(W, y_rul)
    trees = np.array([t.predict(Wt) for t in rf.estimators_])
    P["Random Forest (frozen)"] = (trees.mean(axis=0), *band(trees.mean(axis=0), trees.std(axis=0)))

    search = RandomizedSearchCV(XGBRegressor(objective="reg:squarederror", random_state=0, n_jobs=1), XGB_SEARCH,
                                n_iter=XGB_TRIALS, cv=GroupKFold(n_splits=len(cells)),
                                scoring="neg_root_mean_squared_error", random_state=0, n_jobs=-1).fit(W, y_rul, groups=groups)
    xgb = np.clip(search.predict(Wt), 0, None)

    runs = np.clip(PRED[("Charging", "CNN-LSTM", test_name)]["preds"][:, :, 1], 0, None)   # trained in Part A
    P["CNN-LSTM (frozen)"] = (runs.mean(axis=0), *band(runs.mean(axis=0), runs.std(axis=0)))

    # ---- 3. calibration: replay the twin on each training battery as if it were new ----
    xgb_oof = np.clip(cross_val_predict(XGBRegressor(objective="reg:squarederror", random_state=0, n_jobs=1, **search.best_params_),
                                        W, y_rul, cv=GroupKFold(n_splits=len(cells)), groups=groups), 0, None)
    cal = []
    for i, b in enumerate(cells):
        others = [j for j in range(len(cells)) if j != i]
        prior_i = (intercepts[others].mean(), slopes[others].mean(),
                   max(slopes[others].std(ddof=1), 0.25 * abs(slopes[others].mean())))
        est_i = oof[g == b["Battery"].iloc[0]]                    # SOH estimates from a sensor that never saw this cell
        qi = online_rul(b["k"].values, est_i, prior_i, sigma, sigma_bias, np.random.default_rng(i), np.abs(slopes[others]).min())[SEQ_LEN - 1:]
        keep = b["Before_EOL"].values[SEQ_LEN - 1:]
        cal.append(pd.DataFrame(dict(actual=b["RUL"].values[SEQ_LEN - 1:], online=qi[:, 0],
                                     online_sd=np.maximum((qi[:, 4] - qi[:, 3]) / 2, 1.0), xgb=xgb_oof[groups == i]))[keep])
    cal = pd.concat(cal, ignore_index=True)
    xgb_sd = float(np.sqrt(((cal["actual"] - cal["xgb"]) ** 2).mean()))                       # Tuned XGBoost error on unseen cells
    on_scale = float(np.sqrt((((cal["actual"] - cal["online"]) / cal["online_sd"]) ** 2).mean()))  # how overconfident the trend is
    cal_fused, cal_fused_sd, _ = fuse(cal["online"], on_scale * cal["online_sd"], cal["xgb"], xgb_sd)
    z_x = float(np.quantile(np.abs(cal["actual"] - cal["xgb"]) / xgb_sd, 0.90))
    z_on = float(np.quantile(np.abs(cal["actual"] - cal["online"]) / (on_scale * cal["online_sd"]), 0.90))
    z_f = float(np.quantile(np.abs(cal["actual"] - cal_fused) / cal_fused_sd, 0.90))
    P["Tuned XGBoost (frozen)"] = (xgb, *band(xgb, xgb_sd, z_x))

    # ---- 4. self-updating trend on the test battery, then fusion with Tuned XGBoost ----
    q = online_rul(test["k"].values, soh_est, prior, sigma, sigma_bias, np.random.default_rng(99), np.abs(slopes).min())[SEQ_LEN - 1:]
    on_sd = on_scale * np.maximum((q[:, 4] - q[:, 3]) / 2, 1.0)
    P["Self-updating trend"] = (q[:, 0], *band(q[:, 0], on_sd, z_on))
    fused, fused_sd, weight = fuse(q[:, 0], on_sd, xgb, xgb_sd)
    P["Adaptive Tuned XGBoost twin"] = (fused, *band(fused, fused_sd, z_f))

    meta = meta.assign(SOH_est=soh_est[SEQ_LEN - 1:], Online_weight=weight)
    for name, (p, lo, hi) in P.items():
        meta[name], meta[name + " | low"], meta[name + " | high"] = p, lo, hi
    return meta, list(P)

if have('B2_v4'):
    RESULTS, SOH_ROWS, STREAM = load('B2_v4')
else:
    RESULTS, SOH_ROWS, STREAM = [], [], {}
    for test_name in BATTERIES:
        meta, METHODS = run_twin(test_name)
        STREAM[test_name] = meta
        meta.to_csv(os.path.join(OUT, f"twin_stream_{test_name}.csv"), index=False)
        SOH_ROWS.append(dict(Test_cell=test_name, SOH_RMSE=float(np.sqrt(mean_squared_error(meta["SOH"], meta["SOH_est"]))),
                             SOH_R2=float(r2_score(meta["SOH"], meta["SOH_est"]))))
        pre = meta[meta["Before_EOL"]]                                # RUL is scored before EOL only
        for name in METHODS:
            a, p = pre["RUL"].values, pre[name].values
            RESULTS.append(dict(Method=name, Test_cell=test_name,
                                RUL_RMSE=float(np.sqrt(mean_squared_error(a, p))), RUL_MAE=float(mean_absolute_error(a, p)),
                                RUL_R2=float(r2_score(a, p)),
                                Coverage_90=float(((pre[name + " | low"] <= a) & (a <= pre[name + " | high"])).mean()),
                                Band_width=float((pre[name + " | high"] - pre[name + " | low"]).mean())))
        print("done:", test_name)
    save('B2_v4', (RESULTS, SOH_ROWS, STREAM))
results = pd.DataFrame(RESULTS)
results.to_csv(os.path.join(OUT, "block2_rul_per_cell.csv"), index=False)
average = results.groupby("Method", sort=False).mean(numeric_only=True)
average.to_csv(os.path.join(OUT, "block2_rul_average.csv"))
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
print("\nSOH virtual sensor (linear model on charging features)"); print(pd.DataFrame(SOH_ROWS).set_index("Test_cell"))
print("\nRUL RMSE per test cell (cycles)"); print(results.pivot_table(index="Method", columns="Test_cell", values="RUL_RMSE", sort=False))
print("\nCoverage of the 90% band per test cell (target: about 0.90)"); print(results.pivot_table(index="Method", columns="Test_cell", values="Coverage_90", sort=False))
print("\nAverage over the four test cells"); print(average)

## B3. Figures
Top row: RUL on each unseen battery with the twin's 90% band. Middle row: how much the twin trusts the battery's own trend. Last figure: the SOH virtual sensor.

In [ ]:
STYLE = {"Tuned XGBoost (frozen)": dict(color="tab:blue", ls="--"), "Self-updating trend": dict(color="tab:green", ls=":"),
         "Adaptive Tuned XGBoost twin": dict(color="tab:brown", ls="-")}
fig, axes = plt.subplots(2, 4, figsize=(15, 5.6), gridspec_kw=dict(height_ratios=[3, 1.2]), sharex="col")
for col, test_name in enumerate(BATTERIES):
    m = STREAM[test_name]; ax = axes[0, col]
    ax.plot(m["k"], m["RUL"], color="black", lw=1.8, label="Actual")
    for name, style in STYLE.items():
        ax.plot(m["k"], m[name], lw=1.4, label=name, **style)
    t = "Adaptive Tuned XGBoost twin"
    ax.fill_between(m["k"], m[t + " | low"], m[t + " | high"], color=STYLE[t]["color"], alpha=0.18)
    ax.set_title(f"Test cell {test_name}", fontsize=10); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)
    axes[1, col].plot(m["k"], m["Online_weight"], color="tab:green"); axes[1, col].set_ylim(0, 1)
    axes[1, col].set_xlabel("Discharge cycle"); axes[1, col].grid(True, alpha=0.3)
axes[0, 0].set_ylabel("RUL (cycles)"); axes[1, 0].set_ylabel("Trust in own trend"); axes[0, 0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "Block2_RUL_twin.png"), dpi=300); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(15, 3.1))
for ax, test_name in zip(axes, BATTERIES):
    m = STREAM[test_name]
    ax.plot(m["k"], m["SOH"] * 100, color="black", lw=1.8, label="Actual")
    ax.plot(m["k"], m["SOH_est"] * 100, color="tab:brown", lw=1.3, label="SOH virtual sensor")
    ax.axhline(SOH_EOL * 100, color="gray", ls="--", lw=1)
    ax.set_title(f"Test cell {test_name}", fontsize=10); ax.set_xlabel("Discharge cycle"); ax.grid(True, alpha=0.3)
axes[0].set_ylabel("SOH (%)"); axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "Block2_SOH_sensor.png"), dpi=300); plt.show()
print("Saved figures, tables and per-cycle twin output to", OUT)

# Part C — Second dataset: MIT/Stanford/Toyota LFP cells
**Source:** K. A. Severson, P. M. Attia, et al., "Data-driven prediction of battery cycle life before capacity degradation", *Nature Energy*, 2019. Hosted at https://data.matr.io/1/.

124 commercial LFP/graphite cells (1.1 Ah nominal), fast-charged with different policies, discharged at 4C, at 30 °C, cycled to 80% of nominal capacity. LFP is the chemistry used in grid storage (BESS).

The raw files (several GB) are downloaded to Colab's temporary disk and are **not** stored in your Drive. The prepared table is saved to `MyDrive/FYP_MATR/matr_cycles_v2.csv`. If that file already exists, the download and reading are skipped.

## C1. Setup

In [ ]:
LOCAL_ROOT = "data"         # folder used outside Colab when DATA_ROOT is not set (ignored in Colab)
import os, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
pd.set_option("display.float_format", "{:.4f}".format); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive"
except ImportError:
    ROOT = os.environ.get("DATA_ROOT", LOCAL_ROOT)

RAW = "/content/matr_raw" if os.path.isdir("/content") else os.path.join(ROOT, "matr_raw")   # temporary Colab disk
OUT2 = os.path.join(ROOT, "FYP_MATR")
os.makedirs(RAW, exist_ok=True); os.makedirs(OUT2, exist_ok=True)

FILES = {   # batch prefix: (download link, local file name)
    "b1": ("https://data.matr.io/1/api/v1/file/5c86c0b5fa2ede00015ddf66/download", "2017-05-12_batchdata.mat"),
    "b2": ("https://data.matr.io/1/api/v1/file/5c86bf13fa2ede00015ddd82/download", "2017-06-30_batchdata.mat"),
    "b3": ("https://data.matr.io/1/api/v1/file/5c86bd64fa2ede00015ddbb2/download", "2018-04-12_batchdata.mat"),
}
NOMINAL_AH = 1.1
EOL_AH = 0.88           # 80% of nominal, the dataset's own end-of-life definition
print("Raw files go to:", RAW); print("Results go to:", OUT2)

# ---- saved results: a finished heavy step is stored in the cache folder and loaded next time instead of recomputed ----
import pickle
RERUN_ALL = False                      # set to True to ignore saved results and recompute everything
CACHE_DIR = os.path.join(ROOT, "FYP_cache"); os.makedirs(CACHE_DIR, exist_ok=True)
def have(name): return (not RERUN_ALL) and os.path.exists(os.path.join(CACHE_DIR, name + ".pkl"))
def save(name, obj):
    with open(os.path.join(CACHE_DIR, name + ".pkl"), "wb") as fh: pickle.dump(obj, fh)
def load(name):
    print(f"Loaded saved results '{name}' (set RERUN_ALL = True in the setup cell to recompute)")
    with open(os.path.join(CACHE_DIR, name + ".pkl"), "rb") as fh: return pickle.load(fh)


HAVE = os.path.exists(os.path.join(OUT2, "matr_cycles_v2.csv"))
print("Prepared table already saved:", HAVE)

## C2. Download the raw files (skipped if the prepared table exists)

In [ ]:
import urllib.request
def download(url, path):
    # plain-Python download (works on Windows, Linux and Colab), with progress every 250 MB
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(path + ".part", "wb") as out:
        total, done, mark = int(r.headers.get("Content-Length") or 0), 0, 0
        while True:
            chunk = r.read(4 * 1024 * 1024)
            if not chunk:
                break
            out.write(chunk); done += len(chunk)
            if done - mark >= 250e6:
                mark = done; print(f"   {done / 1e9:.2f} GB" + (f" of {total / 1e9:.2f} GB" if total else ""))
    os.replace(path + ".part", path)

if not HAVE:
    # [download]
    for prefix, (url, fname) in FILES.items():
        path = os.path.join(RAW, fname)
        if not (os.path.exists(path) and h5py.is_hdf5(path)):
            print("Downloading", fname, "...")
            download(url, path)
        ok = os.path.exists(path) and h5py.is_hdf5(path)
        size = os.path.getsize(path) / 1e9 if os.path.exists(path) else 0
        print(f"{fname}: {size:.2f} GB  {'OK' if ok else 'NOT A VALID FILE'}")
        if not ok:
            raise RuntimeError(f"Could not download {fname}. Open https://data.matr.io/1/ in your browser, download the "
                               f"'{fname[:10]}' batch file, upload it to {RAW} with the name {fname}, then run this cell again.")

## C3. Health features from the end of each charge
The cells use different fast-charging policies up to 80% state of charge, so the fast part cannot be compared between cells.
But **every** cell finishes with the same step: 1C constant current up to 3.6 V, then constant voltage. That shared step is where the features come from:

| Feature | Meaning |
|---|---|
| `T_1C_min` | Time spent in the final 1C constant-current step (minutes) |
| `Q_1C_Ah` | Charge delivered during that step |
| `T_CV_min` | Time spent in the final constant-voltage step (minutes) |

These play the same role as the constant-current and constant-voltage features used for the NASA cells.

In [ ]:
NAN_FEATS = dict(T_1C_min=np.nan, Q_1C_Ah=np.nan, T_CV_min=np.nan)

def tail_charge_features(I, V, t, Qc):
    # I (A), V (V), t (min), Qc (Ah, charge added in this cycle), all as stored in the raw file
    if len(t) < 20:
        return NAN_FEATS
    end = int(np.argmax(Qc))                              # charging ends where charge capacity peaks
    if end < 10 or Qc[end] < 0.5:
        return NAN_FEATS
    I, V, t, Qc = I[:end + 1], V[:end + 1], t[:end + 1], Qc[:end + 1]
    dt = np.diff(t, prepend=t[0])
    late = Qc >= 0.85                                     # past the fast-charging part
    cc = late & (I > 0.9) & (I < 1.3) & (V < 3.595)       # final 1C constant-current step
    cv = late & (V >= 3.595) & (I > 0.02) & (I < 1.3)     # final constant-voltage step
    t_cc, t_cv = float(dt[cc].sum()), float(dt[cv].sum())
    if t_cc > 30 or t_cv > 120:                           # a logging gap, not a real charging step
        return NAN_FEATS
    # near end of life the 1C step shrinks towards zero: that is information, so it is kept
    return dict(T_1C_min=t_cc, Q_1C_Ah=float(Qc[cc].max() - Qc[cc].min()) if cc.any() else 0.0, T_CV_min=t_cv)

## C4. Read every cell and cycle (slow; skipped if the prepared table exists)

In [ ]:
if not HAVE:
    # [read]
    SUMMARY_FIELDS = {"QDischarge": "QD_Ah", "QCharge": "QC_Ah", "IR": "IR_ohm", "Tavg": "Tavg", "Tmin": "Tmin",
                      "Tmax": "Tmax", "chargetime": "Chargetime_min"}

    def read_batch(path, prefix):
        cells = {}
        with h5py.File(path, "r") as f:
            batch = f["batch"]
            for i in range(batch["summary"].shape[0]):
                s = f[batch["summary"][i, 0]]
                table = pd.DataFrame({new: np.asarray(s[old][0, :]).flatten() for old, new in SUMMARY_FIELDS.items()})
                policy = f[batch["policy_readable"][i, 0]][()].tobytes()[::2].decode()
                try:
                    life = float(np.asarray(f[batch["cycle_life"][i, 0]][()]).flatten()[0])
                except Exception:
                    life = np.nan
                cyc = f[batch["cycles"][i, 0]]
                refs = {k: cyc[k][:, 0] for k in ("I", "V", "t", "Qc")}
                feats = []
                for j in range(len(refs["I"])):
                    try:
                        feats.append(tail_charge_features(*[np.asarray(f[refs[k][j]][()]).flatten() for k in ("I", "V", "t", "Qc")]))
                    except Exception:
                        feats.append(NAN_FEATS)
                feats = pd.DataFrame(feats)
                n = min(len(table), len(feats))
                cells[f"{prefix}c{i}"] = dict(policy=policy, cycle_life=life, table=pd.concat([table.iloc[:n], feats.iloc[:n]], axis=1))
        return cells

    raw = {}
    for prefix, (url, fname) in FILES.items():
        t0 = time.time()
        raw.update(read_batch(os.path.join(RAW, fname), prefix))
        print(f"{fname}: read, {sum(k.startswith(prefix) for k in raw)} cells, {time.time() - t0:.0f} s")

## C5. Clean up, label and save
Follows the dataset authors' published loading script:
- 5 cells in the first file did not reach 80% capacity and are removed.
- 5 cells were continued in the second file; the two halves are joined.
- 6 noisy cells in the third file are removed.

Then SOH = discharge capacity / 1.1 Ah and RUL = cycles left until end of life. End of life is the first cycle at or below 0.88 Ah.
Many cells were **stopped by the test rig as soon as they reached 0.88 Ah**, so their last recorded cycle is their end of life; the first version of this notebook wrongly dropped those cells.

In [ ]:
FEATURES2 = ["T_1C_min", "Q_1C_Ah", "T_CV_min", "IR_ohm", "Chargetime_min"]
if HAVE:
    matr = pd.read_csv(os.path.join(OUT2, "matr_cycles_v2.csv"))
    cell_table = pd.read_csv(os.path.join(OUT2, "matr_cells_v2.csv"), index_col=0)
    print(f"Loaded prepared table: {matr['Battery'].nunique()} cells, {len(matr)} cycles")
else:
    REMOVE = ["b1c8", "b1c10", "b1c12", "b1c13", "b1c22", "b3c37", "b3c2", "b3c23", "b3c32", "b3c42", "b3c43"]
    JOIN = {"b1c0": "b2c7", "b1c1": "b2c8", "b1c2": "b2c9", "b1c3": "b2c15", "b1c4": "b2c16"}   # first half: second half

    cells = {k: v for k, v in raw.items() if k not in REMOVE}
    for first, second in JOIN.items():
        if first in cells and second in cells:
            cells[first]["table"] = pd.concat([cells[first]["table"], cells[second]["table"]], ignore_index=True)
            cells[first]["cycle_life"] = np.nan
            del cells[second]

    FEATURES2 = ["T_1C_min", "Q_1C_Ah", "T_CV_min", "IR_ohm", "Chargetime_min"]
    frames, dropped, rule = [], [], {"crossed 0.88 Ah": 0, "stopped at 0.88 Ah": 0}
    for name, c in cells.items():
        b = c["table"].copy()
        b.insert(0, "Cycle", np.arange(1, len(b) + 1)); b.insert(0, "Policy", c["policy"]); b.insert(0, "Battery", name)
        b = b[(b["QD_Ah"] > 0.5) & (b["QD_Ah"] < 1.3)].reset_index(drop=True)          # drop empty / faulty cycles
        if len(b) < 50:
            dropped.append(name); continue
        smooth = b["QD_Ah"].rolling(5, center=True, min_periods=1).median()            # ignore single-cycle glitches
        below = b.index[(smooth <= EOL_AH) & (b["Cycle"] > 10)]
        if len(below):
            eol_row = int(below[0]); rule["crossed 0.88 Ah"] += 1
        elif smooth.iloc[-1] <= EOL_AH + 0.02:                                         # test stopped at the 80% criterion
            eol_row = len(b) - 1; rule["stopped at 0.88 Ah"] += 1
        else:
            dropped.append(name); continue
        bad_any = pd.Series(False, index=b.index)
        for f in FEATURES2:                                                            # replace glitches by interpolation
            med = b[f].rolling(9, center=True, min_periods=1).median()
            bad = b[f].isna() | ((b[f] - med).abs() > 0.30 * med.abs() + 0.10 * abs(b[f].median()))
            b.loc[bad, f] = np.nan
            bad_any |= bad
        b["Feature_filled"] = bad_any
        b[FEATURES2] = b[FEATURES2].interpolate(limit_direction="both")
        if b[FEATURES2].isna().any().any() or bad_any.mean() > 0.30:
            dropped.append(name); continue
        b["k"] = np.arange(1, len(b) + 1)
        b["SOH"] = b["QD_Ah"] / NOMINAL_AH
        b["EOL_k"] = eol_row + 1
        b["Cycle_life_authors"] = c.get("cycle_life", np.nan)
        b["RUL"] = (b["EOL_k"] - b["k"]).clip(lower=0)
        b["Before_EOL"] = b["k"] <= b["EOL_k"]
        frames.append(b)

    matr = pd.concat(frames, ignore_index=True)
    matr.to_csv(os.path.join(OUT2, "matr_cycles_v2.csv"), index=False)
    cell_table = matr.groupby("Battery", sort=False).agg(Policy=("Policy", "first"), Cycles=("k", "max"), EOL_cycle=("EOL_k", "first"),
                                                         First_cap_Ah=("QD_Ah", "first"), Last_cap_Ah=("QD_Ah", "last"),
                                                         Features_filled_pct=("Feature_filled", lambda s: 100 * s.mean()))
    cell_table.to_csv(os.path.join(OUT2, "matr_cells_v2.csv"))
    print(f"{matr['Battery'].nunique()} cells kept, {len(matr)} cycles in total. Dropped: {dropped}")
    print("How end of life was found:", rule)
    check = matr.groupby("Battery").agg(mine=("EOL_k", "first"), authors=("Cycle_life_authors", "first")).dropna()
    print(f"Difference from the authors' cycle life, median over {len(check)} cells: {(check['mine'] - check['authors']).abs().median():.0f} cycles")
    print("Cells per file:", matr.groupby(matr["Battery"].str[:2])["Battery"].nunique().to_dict())
    print("\nEnd-of-life cycle across cells:"); print(cell_table["EOL_cycle"].describe()[["min", "25%", "50%", "75%", "max"]].to_string())
    print("\nFeatures that needed filling, % of cycles per cell:"); print(cell_table["Features_filled_pct"].describe()[["min", "50%", "max"]].to_string())
    print("\nSaved:", os.path.join(OUT2, "matr_cycles_v2.csv"))
print("\nEnd-of-life cycle across cells:"); print(cell_table["EOL_cycle"].describe()[["min", "25%", "50%", "75%", "max"]].to_string())

## C6. Do the features track health in this dataset too?

In [ ]:
cols = FEATURES2
corr = matr.groupby("Battery")[cols + ["QD_Ah"]].apply(lambda b: b[cols].corrwith(b["QD_Ah"]))
print("Correlation with capacity across cells (median, and the middle half of cells)")
print(corr.describe().loc[["25%", "50%", "75%"]].T.rename(columns={"25%": "lower quarter", "50%": "median", "75%": "upper quarter"}))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for name, b in matr.groupby("Battery"):
    axes[0].plot(b["k"], b["QD_Ah"], lw=0.5, alpha=0.6)
    axes[1].plot(b["k"], b["T_1C_min"], lw=0.5, alpha=0.6)
    axes[2].plot(b["QD_Ah"], b["T_1C_min"], lw=0.5, alpha=0.6)
axes[0].axhline(EOL_AH, color="black", ls="--", lw=1)
axes[0].set_xlabel("Cycle"); axes[0].set_ylabel("Discharge capacity (Ah)")
axes[1].set_xlabel("Cycle"); axes[1].set_ylabel("Final 1C step time (min)")
axes[2].set_xlabel("Discharge capacity (Ah)"); axes[2].set_ylabel("Final 1C step time (min)")
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT2, "matr_overview.png"), dpi=200); plt.show()
cell_table.head(10)

# Part D — Second dataset: model comparison
The same five models as Part A, now on the LFP cells.

- **Inputs:** the five charging-side features (final 1C step time and charge, constant-voltage time, internal resistance, total charge time), as a 20-cycle window.
- **Test:** 5-fold cross-validation **grouped by cell**. Every cell is predicted by models that never saw any of its cycles.
- **Tuned XGBoost:** its settings are searched inside each training fold with cross-validation grouped by cell, separately for SOH and RUL, so tuning never sees the test cells. The chosen settings are saved.
- **Speed:** every 3rd cycle is used. Deep models average five seeds.

## D1. Model definitions (repeated here so Part D can run without Part A)

In [ ]:
import random
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_val_predict
from xgboost import XGBRegressor
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEQ_LEN = 20
N_SEEDS = 5          # training runs averaged for each deep model
EPOCHS = 100

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

def build_model(n_features, kind="CNN-LSTM"):
    inp = keras.Input((SEQ_LEN, n_features))
    x = inp
    if kind == "CNN-LSTM":
        x = layers.Conv1D(64, 3, padding="causal", activation="relu")(x)
        x = layers.Conv1D(64, 3, padding="causal", activation="relu")(x)
    x = layers.LSTM(64)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation="relu")(x)
    m = keras.Model(inp, layers.Dense(2)(x))
    m.compile(optimizer="adam", loss="mse")
    return m

def fit_model(m, X, y, Xv, yv):
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    m.fit(X, y, validation_data=(Xv, yv), epochs=EPOCHS, batch_size=16, callbacks=[stop], verbose=0)
    return m

def fit_fast(m, X, y, Xv, yv):
    # larger batches and fewer epochs than Part A: this dataset has far more cycles
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    m.fit(X, y, validation_data=(Xv, yv), epochs=40, batch_size=256, callbacks=[stop], verbose=0)
    return m

## D2. Build the 20-cycle windows

In [ ]:
STEP = 3                                   # use every 3rd cycle
seqs, targets, groups2, pre2, cyc2 = [], [], [], [], []
for gi, (name, b) in enumerate(matr.groupby("Battery", sort=False)):
    X = b[FEATURES2].values; idx = np.arange(SEQ_LEN - 1, len(b), STEP)
    seqs.append(np.array([X[i - SEQ_LEN + 1:i + 1] for i in idx]))
    targets.append(b[["SOH", "RUL"]].values[idx]); pre2.append(b["Before_EOL"].values[idx].astype(bool))
    groups2.append(np.full(len(idx), gi)); cyc2.append(b["k"].values[idx])
SEQ2, Y2, G2, PRE2, K2 = (np.concatenate(v) for v in (seqs, targets, groups2, pre2, cyc2))
CELL_NAMES2 = list(matr.groupby("Battery", sort=False).groups)
print("Windows:", SEQ2.shape, "| cells:", len(CELL_NAMES2))

## D3. Grouped 5-fold comparison
Takes roughly 30 minutes. It prints a line as each fold finishes.

In [ ]:
MODELS2 = ["Linear", "Random Forest", "Tuned XGBoost", "LSTM", "CNN-LSTM"]
XGB_SEARCH2 = dict(n_estimators=[200, 400, 600], max_depth=[3, 4, 6], learning_rate=[0.03, 0.05, 0.1],
                   subsample=[0.6, 0.8, 1.0], colsample_bytree=[0.6, 0.8, 1.0], min_child_weight=[1, 5, 10])
XGB_TRIALS2 = 6
new_xgb = lambda **kw: XGBRegressor(objective="reg:squarederror", tree_method="hist", random_state=0, n_jobs=-1, **kw)
DEEP_SEEDS = 5                             # each deep model is the average of this many training runs
nf = len(FEATURES2)
summary2 = lambda S: np.concatenate([S[:, -1, :], S.mean(axis=1), S[:, -1, :] - S[:, 0, :]], axis=1)
W_raw = summary2(SEQ2)
if have('D3_v4'):
    PRED2, XGB_PARAMS2 = load('D3_v4')
else:
    XGB_PARAMS2 = []
    PRED2 = {m: np.full(Y2.shape, np.nan) for m in MODELS2}

    for fold, (tr, te) in enumerate(GroupKFold(n_splits=5).split(SEQ2, groups=G2)):
        t0 = time.time()
        xs = StandardScaler().fit(SEQ2[tr].reshape(-1, nf)); ys = StandardScaler().fit(Y2[tr])
        Str, Ste = (xs.transform(SEQ2[i].reshape(-1, nf)).reshape(-1, SEQ_LEN, nf) for i in (tr, te))
        ytr = ys.transform(Y2[tr]); Wtr, Wte = summary2(Str), summary2(Ste)

        PRED2["Linear"][te] = ys.inverse_transform(Ridge(alpha=1.0).fit(Str[:, -1, :], ytr).predict(Ste[:, -1, :]))
        rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(Wtr, ytr)
        PRED2["Random Forest"][te] = ys.inverse_transform(rf.predict(Wte))
        out, best = [], {}
        for j, target in enumerate(["SOH", "RUL"]):               # trees need no scaling: raw features and raw targets
            search = RandomizedSearchCV(new_xgb(), XGB_SEARCH2, n_iter=XGB_TRIALS2, cv=GroupKFold(n_splits=3),
                                        scoring="neg_root_mean_squared_error", random_state=fold).fit(W_raw[tr], Y2[tr, j], groups=G2[tr])
            out.append(search.predict(W_raw[te])); best[target] = search.best_params_
        PRED2["Tuned XGBoost"][te] = np.column_stack(out); XGB_PARAMS2.append(best)
        set_seed(0)
        val_cells = np.random.choice(np.unique(G2[tr]), max(1, len(np.unique(G2[tr])) // 7), replace=False)   # validation by cell
        va = np.isin(G2[tr], val_cells)
        for kind in ["LSTM", "CNN-LSTM"]:
            runs = []
            for seed in range(DEEP_SEEDS):
                set_seed(seed)
                m = fit_fast(build_model(nf, kind), Str[~va], ytr[~va], Str[va], ytr[va])
                runs.append(ys.inverse_transform(m.predict(Ste, verbose=0, batch_size=1024)))
            PRED2[kind][te] = np.mean(runs, axis=0)
        print(f"fold {fold + 1}/5 done in {time.time() - t0:.0f} s")
    save('D3_v4', (PRED2, XGB_PARAMS2))

rows = []
for mname in MODELS2:
    p = PRED2[mname].copy(); p[:, 1] = np.clip(p[:, 1], 0, None)
    per_cell = [np.sqrt(mean_squared_error(Y2[(G2 == g) & PRE2, 1], p[(G2 == g) & PRE2, 1])) for g in np.unique(G2)]
    rows.append(dict(Model=mname,
                     SOH_RMSE=np.sqrt(mean_squared_error(Y2[:, 0], p[:, 0])), SOH_R2=r2_score(Y2[:, 0], p[:, 0]),
                     RUL_RMSE=np.sqrt(mean_squared_error(Y2[PRE2, 1], p[PRE2, 1])), RUL_MAE=mean_absolute_error(Y2[PRE2, 1], p[PRE2, 1]),
                     RUL_R2=r2_score(Y2[PRE2, 1], p[PRE2, 1]), RUL_RMSE_median_cell=np.median(per_cell), RUL_RMSE_worst_cell=np.max(per_cell)))
results2 = pd.DataFrame(rows).set_index("Model")
results2.to_csv(os.path.join(OUT2, "matr_model_comparison.csv"))
pd.DataFrame([dict(Fold=q + 1, Target=t, **p) for q, b in enumerate(XGB_PARAMS2) for t, p in b.items()]).to_csv(os.path.join(OUT2, "matr_xgboost_best_hyperparameters.csv"), index=False)
print("\nSecond dataset, every cell predicted by models that never saw it")
print(results2)

## D4. Figures: three example cells (shortest-lived, median, longest-lived)

In [ ]:
life = matr.groupby("Battery", sort=False)["EOL_k"].first().values
picks = [int(np.argmin(life)), int(np.argsort(life)[len(life) // 2]), int(np.argmax(life))]
STYLE2 = {"Tuned XGBoost": dict(color="tab:green", ls=":"), "LSTM": dict(color="tab:blue", ls="--"), "CNN-LSTM": dict(color="tab:brown", ls="-")}
fig, axes = plt.subplots(2, 3, figsize=(14, 6.2))
for col, g in enumerate(picks):
    sel = G2 == g
    for row, (j, ylabel, scale) in enumerate([(0, "SOH (%)", 100), (1, "RUL (cycles)", 1)]):
        ax = axes[row, col]
        ax.plot(K2[sel], Y2[sel, j] * scale, color="black", lw=1.8, label="Actual")
        for mname, style in STYLE2.items():
            ax.plot(K2[sel], np.clip(PRED2[mname][sel, j], 0, None) * scale, lw=1.3, label=mname, **style)
        ax.set_xlabel("Cycle"); ax.set_ylabel(ylabel); ax.grid(True, alpha=0.3)
    axes[0, col].set_title(f"{CELL_NAMES2[g]} (end of life at cycle {life[g]})", fontsize=10)
axes[0, 0].legend(frameon=False, fontsize=8); plt.tight_layout()
plt.savefig(os.path.join(OUT2, "matr_examples.png"), dpi=200); plt.show()

# Part E — Second dataset: self-updating twin and confidence bands
Built on the Tuned XGBoost RUL predictions from Part D, for the 124 LFP cells, all tested on cells never seen in training (same 5 folds as Part D):

1. **Tuned XGBoost.** The RUL predictions and the per-fold settings come from Part D.
2. **Self-updating twin filter.** Every prediction implies an end-of-life cycle (current cycle + predicted RUL). The twin keeps a running belief about that end-of-life cycle and updates it each time a new prediction arrives, trusting predictions more as the cell gets closer to end of life. This removes cycle-to-cycle jitter. It does **not** remove a bias that is present in every prediction for a cell.
3. **Calibrated 90% bands.** Inside each training fold the method is replayed on held-out training cells, and the band is widened until it would have covered 90% of those cases. The band is proportional to the predicted RUL, so it narrows as end of life approaches.

## E1. Run
Takes a few minutes the first time.

In [ ]:
C0 = 20.0                                   # floor (cycles) so the relative band does not collapse at end of life
Q_GRID = [0.0, 0.5, 2.0, 8.0, 32.0]         # how fast the twin's belief is allowed to drift (variance per cycle)

def eol_filter(k, p, rel, q_var):
    # scalar Kalman filter on the end-of-life cycle of ONE cell; k = cycle numbers, p = RUL predictions at those cycles
    x, P, out = k[0] + p[0], (rel * (p[0] + C0)) ** 2, [max(p[0], 0.0)]
    for i in range(1, len(k)):
        P += q_var * (k[i] - k[i - 1])
        R = (rel * (max(p[i], 0.0) + C0)) ** 2            # predictions near end of life are trusted more
        gain = P / (P + R)
        x += gain * (k[i] + p[i] - x); P *= 1 - gain
        out.append(max(x - k[i], 0.0))
    return np.array(out)

def filter_cells(k, p, g, rel, q_var):
    out = np.empty(len(p))
    for cell in np.unique(g):
        m = g == cell
        out[m] = eol_filter(k[m], p[m], rel, q_var)
    return out

if have('E1_v2'):
    E = load('E1_v2')
else:
    W2 = summary2(SEQ2)                                    # trees need no scaling
    E = dict(xgb=np.full(len(W2), np.nan), twin=np.full(len(W2), np.nan), xgb_half=np.full(len(W2), np.nan),
             twin_half=np.full(len(W2), np.nan), params=[])
    for fold, (tr, te) in enumerate(GroupKFold(n_splits=5).split(W2, groups=G2)):
        t0 = time.time()
        params = XGB_PARAMS2[fold]["RUL"]                       # the settings Part D chose for this fold
        E["params"].append(dict(Fold=fold + 1, **params))
        p_te = np.clip(PRED2["Tuned XGBoost"][te, 1], 0, None)
        # replay on held-out training cells: predictions each training cell would get from a model that never saw it
        p_cal = np.clip(cross_val_predict(new_xgb(**params), W2[tr], Y2[tr, 1], cv=GroupKFold(n_splits=4), groups=G2[tr]), 0, None)
        y_cal, k_cal, g_cal, pre_cal = Y2[tr, 1], K2[tr], G2[tr], PRE2[tr]
        rel = float(np.sqrt((((y_cal - p_cal) / (p_cal + C0)) ** 2)[pre_cal].mean()))
        rmse_q = [np.sqrt(((y_cal - filter_cells(k_cal, p_cal, g_cal, rel, q))[pre_cal] ** 2).mean()) for q in Q_GRID]
        q_var = Q_GRID[int(np.argmin(rmse_q))]
        f_cal = filter_cells(k_cal, p_cal, g_cal, rel, q_var)
        z_x = float(np.quantile((np.abs(y_cal - p_cal) / (p_cal + C0))[pre_cal], 0.90))
        z_f = float(np.quantile((np.abs(y_cal - f_cal) / (f_cal + C0))[pre_cal], 0.90))
        f_te = filter_cells(K2[te], p_te, G2[te], rel, q_var)
        E["xgb"][te], E["twin"][te] = p_te, f_te
        E["xgb_half"][te], E["twin_half"][te] = z_x * (p_te + C0), z_f * (f_te + C0)
        print(f"fold {fold + 1}/5 done in {time.time() - t0:.0f} s | drift setting {q_var} | band multiplier Tuned XGBoost {z_x:.2f}, twin {z_f:.2f}")
    save('E1_v2', E)

def rul_row(name, p, half=None):
    y = Y2[:, 1]
    per_cell = [np.sqrt(mean_squared_error(y[(G2 == g) & PRE2], p[(G2 == g) & PRE2])) for g in np.unique(G2)]
    row = dict(Method=name, RUL_RMSE=np.sqrt(mean_squared_error(y[PRE2], p[PRE2])), RUL_MAE=mean_absolute_error(y[PRE2], p[PRE2]),
               RUL_R2=r2_score(y[PRE2], p[PRE2]), RMSE_median_cell=np.median(per_cell), RMSE_worst_cell=np.max(per_cell))
    if half is not None:
        inside = (y >= np.clip(p - half, 0, None)) & (y <= p + half)
        row.update(Coverage_90=inside[PRE2].mean(), Band_width=(2 * half)[PRE2].mean())
    return row

resultsE = pd.DataFrame([rul_row("Tuned XGBoost", E["xgb"], E["xgb_half"]),
                         rul_row("Self-updating twin (Tuned XGBoost + filter)", E["twin"], E["twin_half"])]).set_index("Method")
resultsE.to_csv(os.path.join(OUT2, "matr_twin_results.csv"))
print("\nSecond dataset, RUL on cells never seen in training"); print(resultsE)
third = np.digitize(Y2[:, 1] / np.maximum(Y2[:, 1] + K2, 1), [1 / 3, 2 / 3])        # 2 = early life, 0 = late life
stage = pd.DataFrame({name: [np.sqrt(((Y2[:, 1] - p)[PRE2 & (third == t)] ** 2).mean()) for t in (2, 1, 0)]
                      for name, p in [("Tuned XGBoost", E["xgb"]), ("Self-updating twin", E["twin"])]},
                     index=["Early life", "Mid life", "Late life"])
print("\nRUL RMSE by stage of life (cycles)"); print(stage)

## E2. Figures: three example cells with the twin's 90% band

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, g in zip(axes, picks):
    sel = G2 == g
    ax.plot(K2[sel], Y2[sel, 1], color="black", lw=1.8, label="Actual")
    ax.plot(K2[sel], E["xgb"][sel], color="tab:green", ls=":", lw=1.2, label="Tuned XGBoost")
    ax.plot(K2[sel], E["twin"][sel], color="tab:brown", lw=1.5, label="Self-updating twin")
    ax.fill_between(K2[sel], np.clip(E["twin"][sel] - E["twin_half"][sel], 0, None), E["twin"][sel] + E["twin_half"][sel],
                    color="tab:brown", alpha=0.18, label="90% band")
    ax.set_title(f"{CELL_NAMES2[g]} (end of life at cycle {life[g]})", fontsize=10)
    ax.set_xlabel("Cycle"); ax.set_ylabel("RUL (cycles)"); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)
axes[0].legend(frameon=False, fontsize=8); plt.tight_layout()
plt.savefig(os.path.join(OUT2, "matr_twin_examples.png"), dpi=200); plt.show()

# Part F — SOC estimation on the NASA cells
SOC (state of charge) says how full the battery is **right now**, so it is estimated from the moment-by-moment discharge curve, not from one row per cycle.

**Reference SOC.** Within each discharge, the charge taken out so far is counted; SOC runs from 100% at the start to 0% when the discharge ends.

**Four methods, each tested on a cell never seen in training:**

| Method | Idea |
|---|---|
| Charge counting, fixed capacity | Counts charge and divides by the capacity measured in the cell's first cycle. Ignores ageing. |
| Charge counting + twin SOH | Same counting, but divides by the capacity the twin estimates **before the discharge starts**, from the preceding charge (the SOH virtual sensor). |
| Tuned XGBoost from voltage | Estimates SOC from voltage, current and temperature at that moment. Needs no charge counting. Settings are searched with cross-validation grouped by battery. |
| Tuned XGBoost from voltage + twin SOH | Same, with the twin's SOH estimate as an extra input. |

**Limitation.** These cells are discharged at a constant current, which makes SOC easy to estimate. Good numbers here do not show how the methods behave under a changing load; that needs the hardware stage.

This part needs Part A to have run (it reuses the NASA data and charging features).

## F1. Run

In [ ]:
from sklearn.model_selection import cross_val_predict, RandomizedSearchCV

def discharge_curves(name):
    cycles = loadmat(os.path.join(battery_path, name + ".mat"))[name][0, 0]["cycle"]
    out, k = [], 0
    for idx in range(cycles.shape[1]):
        c = cycles[0, idx]
        if str(c["type"][0]) != "discharge":
            continue
        d = c["data"][0, 0]
        cap = arr(d, "Capacity")
        if len(cap) == 0 or not np.isfinite(cap[0]):
            continue
        k += 1                                                   # same cycle numbering as the Part A table
        t, v, i, temp = arr(d, "Time"), arr(d, "Voltage_measured"), arr(d, "Current_measured"), arr(d, "Temperature_measured")
        on = np.where(i < -0.5)[0]                               # samples where the load is actually drawing current
        if len(on) < 20:
            continue
        a, z = on[0], on[-1] + 1
        t, v, i, temp = t[a:z], v[a:z], i[a:z], temp[a:z]
        q = np.concatenate([[0.0], np.cumsum(-(i[:-1] + i[1:]) / 2 * np.diff(t)) / 3600])
        out.append(pd.DataFrame(dict(Battery=name, k=k, Time_s=t - t[0], V=v, I=i, T=temp, Q_Ah=q, SOC=1 - q / q[-1], Q_end_Ah=q[-1])))
    return pd.concat(out, ignore_index=True)

SOC_METHODS = ["Charge counting, fixed capacity", "Charge counting + twin SOH", "Tuned XGBoost from voltage", "Tuned XGBoost from voltage + twin SOH"]
SOC_SEARCH = dict(n_estimators=[200, 300, 500], max_depth=[3, 5, 7], learning_rate=[0.03, 0.08, 0.15], subsample=[0.7, 0.9, 1.0], min_child_weight=[1, 5, 20])
SOC_TRIALS = 5
if have('F1_v2'):
    soc_test = load('F1_v2')
else:
    curves = pd.concat([discharge_curves(n) for n in BATTERIES], ignore_index=True)
    print("Discharge samples:", len(curves))
    def soc_tuned(train, cols):
        # Tuned XGBoost for SOC: the search is grouped by battery, so no battery is in both the fitting and the checking part
        base = XGBRegressor(objective="reg:squarederror", tree_method="hist", random_state=0, n_jobs=-1)
        return RandomizedSearchCV(base, SOC_SEARCH, n_iter=SOC_TRIALS, cv=GroupKFold(n_splits=train["Battery"].nunique()),
                                  scoring="neg_root_mean_squared_error", random_state=0).fit(train[cols], train["SOC"], groups=train["Battery"])
    parts = []
    for test_name in BATTERIES:
        tr_c, te_c = df[df["Battery"] != test_name], df[df["Battery"] == test_name]
        xs = StandardScaler().fit(tr_c[CHARGE_FEATURES].values)
        # twin SOH sensor: for training cells use estimates from a sensor that did not see that cell
        soh_tr = cross_val_predict(Ridge(alpha=1.0), xs.transform(tr_c[CHARGE_FEATURES].values), tr_c["SOH"].values,
                                   cv=GroupKFold(n_splits=tr_c["Battery"].nunique()), groups=tr_c["Battery"].values)
        sensor = Ridge(alpha=1.0).fit(xs.transform(tr_c[CHARGE_FEATURES].values), tr_c["SOH"].values)
        soh_map = pd.concat([tr_c.assign(SOH_est=soh_tr), te_c.assign(SOH_est=sensor.predict(xs.transform(te_c[CHARGE_FEATURES].values)))])
        cur = curves.merge(soh_map[["Battery", "k", "SOH_est"]], on=["Battery", "k"], how="inner")
        tr, te = cur[cur["Battery"] != test_name].iloc[::2], cur[cur["Battery"] == test_name].copy()     # every 2nd training sample
        first_cap = te.loc[te["k"] == te["k"].min(), "Q_end_Ah"].iloc[0]
        te[SOC_METHODS[0]] = np.clip(1 - te["Q_Ah"] / first_cap, 0, 1)
        te[SOC_METHODS[1]] = np.clip(1 - te["Q_Ah"] / (te["SOH_est"] * RATED_CAPACITY_AH), 0, 1)
        te[SOC_METHODS[2]] = np.clip(soc_tuned(tr, ["V", "I", "T"]).predict(te[["V", "I", "T"]]), 0, 1)
        te[SOC_METHODS[3]] = np.clip(soc_tuned(tr, ["V", "I", "T", "SOH_est"]).predict(te[["V", "I", "T", "SOH_est"]]), 0, 1)
        parts.append(te); print("done:", test_name)
    soc_test = pd.concat(parts, ignore_index=True)
    save('F1_v2', soc_test)

err = lambda d, m: 100 * np.sqrt(((d[m] - d["SOC"]) ** 2).mean())            # RMSE in SOC percentage points
soc_cell = pd.DataFrame({m: soc_test.groupby("Battery").apply(lambda d: err(d, m)) for m in SOC_METHODS}).T
soc_cell["Average"] = soc_cell.mean(axis=1)
frac = soc_test["k"] / soc_test.groupby("Battery")["k"].transform("max")
soc_stage = pd.DataFrame({"First third of life": [err(soc_test[frac <= 1 / 3], m) for m in SOC_METHODS],
                          "Last third of life": [err(soc_test[frac > 2 / 3], m) for m in SOC_METHODS]}, index=SOC_METHODS)
soc_cell.to_csv(os.path.join(OUT, "soc_rmse_per_cell.csv")); soc_stage.to_csv(os.path.join(OUT, "soc_rmse_by_life_stage.csv"))
print("\nSOC RMSE per test cell (percentage points of SOC)"); print(soc_cell)
print("\nSOC RMSE early and late in life (percentage points of SOC)"); print(soc_stage)

## F2. Figures
Top: one late-life discharge of each test cell. Bottom: SOC error per cycle across the cell's life.

In [ ]:
SOC_STYLE = {SOC_METHODS[0]: dict(color="tab:gray", ls="--"), SOC_METHODS[1]: dict(color="tab:brown", ls="-"),
             SOC_METHODS[2]: dict(color="tab:blue", ls=":"), SOC_METHODS[3]: dict(color="tab:green", ls="-.")}
fig, axes = plt.subplots(2, 4, figsize=(15, 6.2))
for col, name in enumerate(BATTERIES):
    b = soc_test[soc_test["Battery"] == name]
    late = b[b["k"] == int(0.9 * b["k"].max())]
    ax = axes[0, col]
    ax.plot(late["Time_s"] / 60, late["SOC"] * 100, color="black", lw=2, label="Reference")
    for m, style in SOC_STYLE.items():
        ax.plot(late["Time_s"] / 60, late[m] * 100, lw=1.4, label=m, **style)
    ax.set_title(f"{name}, discharge cycle {int(late['k'].iloc[0])}", fontsize=10); ax.set_xlabel("Time (min)"); ax.grid(True, alpha=0.3)
    ax2 = axes[1, col]
    for m, style in SOC_STYLE.items():
        per = b.groupby("k").apply(lambda d: err(d, m))
        ax2.plot(per.index, per.values, lw=1.2, **style)
    ax2.set_xlabel("Discharge cycle"); ax2.grid(True, alpha=0.3)
axes[0, 0].set_ylabel("SOC (%)"); axes[1, 0].set_ylabel("SOC RMSE (points)"); axes[0, 0].legend(frameon=False, fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "SOC_estimation.png"), dpi=300); plt.show()

# Part G — Monitoring dashboard (digital twin interface)
Builds one self-contained web page, `twin_dashboard.html`, from the results above and saves it to your Drive. Open it in any browser; it needs no internet and no Python.

**What it shows**
- The four NASA cylindrical cells side by side as one small battery group, replayed cycle by cycle (Play, or drag the slider). Each cell is drawn as a battery cylinder whose fill height is its estimated health.
- Group summary: health, usable capacity, weakest cell, shortest remaining life, alerts.
- Per cell: status, SOH estimate, RUL with its 90% band, peak temperature. Click a cell to see its charts.
- Charts: SOH (twin estimate against the measured value), RUL with band, and SOC during the current discharge (ageing-aware twin against fixed-capacity counting).
- An alerts and maintenance log.
- A second view, **LFP cells**: four cells from the second dataset, from short-lived to long-lived, with SOH and RUL from Parts D and E. Switch views with the buttons at the top.

**What it is, precisely.** A replay of recorded test data. Data flows from battery to twin only, which the literature calls a *digital shadow*. The measured SOH and actual RUL are drawn for comparison; a live system would not have them. Alert thresholds are illustrative settings.

This part needs Parts A, B and F. The LFP view also needs Parts C, D and E; without them the page shows the NASA view only.

## G1. Build and show the dashboard

In [ ]:
DASHBOARD_TEMPLATE = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Battery Digital Twin Monitor</title>
<style>
:root{color-scheme:light;--page:#ffffff;--surface:#ffffff;--ink:#17212b;--ink2:#4d5966;--ink3:#7b8794;--grid:#e9edf1;--axis:#c5ccd4;
  --border:#e1e6eb;--wrap:#1c2a3a;--steel:#c9d0d8;--s1:#2a78d6;--s2:#eb6834;--good:#0ca30c;--warning:#fab219;--serious:#ec835a;--critical:#d03b3b;
  --num:Bahnschrift,"DIN Alternate","Roboto Condensed","Arial Narrow",system-ui,sans-serif}
*{box-sizing:border-box}
body{margin:0;background:var(--page);color:var(--ink);font:14px/1.45 system-ui,-apple-system,"Segoe UI",sans-serif;font-variant-numeric:tabular-nums}
.wrap{max-width:1240px;margin:0 auto;padding:22px 18px 28px}
header{display:flex;flex-wrap:wrap;gap:14px 28px;align-items:flex-end;justify-content:space-between;margin-bottom:18px}
h1{font:600 30px/1.1 var(--num);letter-spacing:.2px;margin:0 0 6px}
.sub{color:var(--ink2);font-size:13.5px;max-width:64ch}
.sets{display:inline-flex;border:1px solid var(--axis);border-radius:9px;overflow:hidden}
.sets:empty{display:none}
.sets button{font:inherit;border:0;background:#fff;color:var(--ink2);padding:7px 14px;cursor:pointer}
.sets button+button{border-left:1px solid var(--axis)}
.sets button[aria-pressed=true]{background:var(--wrap);color:#fff;font-weight:600}
.sets button:focus-visible{outline:2px solid var(--s1);outline-offset:-2px}
.replay{display:flex;flex-wrap:wrap;gap:10px;align-items:center}
.replay button,.replay select{font:inherit;color:var(--ink);background:#fff;border:1px solid var(--axis);border-radius:8px;padding:6px 12px;cursor:pointer}
.replay button{min-width:78px;font-weight:600;background:var(--wrap);color:#fff;border-color:var(--wrap)}
.replay button:focus-visible,.cellbtn:focus-visible,.replay select:focus-visible{outline:2px solid var(--s1);outline-offset:2px}
.replay input[type=range]{width:min(300px,46vw);accent-color:var(--wrap)}
.replay .now{min-width:118px;color:var(--ink2);font-family:var(--num);font-size:15px}
.top{display:grid;grid-template-columns:minmax(0,1fr) 300px;gap:0;border:1px solid var(--border);border-radius:16px;margin-bottom:16px;overflow:hidden}
@media (max-width:900px){.top{grid-template-columns:1fr}.summary{border-left:0!important;border-top:1px solid var(--border)}}
.bench{padding:22px 16px 18px;min-width:0;display:flex;align-items:center}
.cells{width:100%;display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:6px}
.cellbtn{font:inherit;color:inherit;background:none;border:0;padding:0 4px 4px;cursor:pointer;text-align:center;border-radius:12px;min-width:0}
.cellbtn svg{width:112px;height:214px;display:block;margin:0 auto;transition:transform .18s ease}
.cellbtn:hover svg{transform:translateY(-3px)}
.cellbtn[aria-pressed=true] svg{transform:translateY(-6px)}
@media (prefers-reduced-motion:reduce){.cellbtn svg{transition:none}}
.plate{height:8px;border-radius:4px;background:var(--steel);margin:-4px 14px 12px;position:relative}
.cellbtn[aria-pressed=true] .plate{background:var(--s1)}
.cname{font:600 17px/1.2 var(--num);letter-spacing:.3px}
.big{font:600 27px/1.15 var(--num);margin-top:6px}
.big small{font:400 12.5px system-ui,sans-serif;color:var(--ink3);margin-left:3px}
.line{color:var(--ink2);font-size:12.5px;margin-top:2px}
.line b{color:var(--ink);font-weight:600}
.pill{display:inline-flex;align-items:center;gap:5px;border:1px solid var(--border);border-radius:999px;padding:1px 9px 1px 7px;font-size:12px;color:var(--ink);white-space:nowrap;background:#fff}
.pill i{font-style:normal;font-size:11px}
.summary{border-left:1px solid var(--border);padding:6px 18px}
.stat{padding:11px 0;border-top:1px solid var(--grid)}
.stat:first-child{border-top:0}
.stat .lab{color:var(--ink2);font-size:12.5px}
.stat .val{font:600 24px/1.15 var(--num);margin-top:1px}
.stat .val small{font:400 13px system-ui,sans-serif;color:var(--ink2)}
.stat .note{color:var(--ink3);font-size:12px}
.grid{display:grid;gap:14px}
.charts{grid-template-columns:repeat(auto-fit,minmax(min(440px,100%),1fr));margin-bottom:14px}
.card{background:#fff;border:1px solid var(--border);border-radius:14px;padding:14px 16px;min-width:0}
h2{font:600 16px/1.2 var(--num);letter-spacing:.2px;margin:0}
.chead{display:flex;flex-wrap:wrap;justify-content:space-between;gap:4px 14px;align-items:baseline;margin-bottom:4px}
.legend{display:flex;flex-wrap:wrap;gap:4px 14px;color:var(--ink2);font-size:12px}
.legend span{display:inline-flex;align-items:center;gap:5px}
.key{width:14px;height:0;border-top:2px solid;display:inline-block}
.key.band{height:9px;border:0;opacity:.25;border-radius:2px}
.plot{position:relative}
.plot svg{display:block;width:100%;height:230px}
.plot svg text{fill:var(--ink3);font-size:11px}
.tip{position:absolute;pointer-events:none;background:#fff;border:1px solid var(--axis);border-radius:8px;padding:6px 9px;font-size:12px;box-shadow:0 6px 18px rgba(23,33,43,.12);display:none;white-space:nowrap;z-index:2}
.tip b{font-weight:600}.tip div{display:flex;gap:10px;justify-content:space-between}
.alerts{max-height:262px;overflow:auto;margin:0;padding:0;list-style:none}
.alerts li{display:grid;grid-template-columns:auto auto 1fr;gap:4px 10px;align-items:center;padding:7px 0;border-top:1px solid var(--grid);font-size:13px}
.alerts li:first-child{border-top:0}
.alerts .when{color:var(--ink3);font-size:12px;white-space:nowrap}
.empty{color:var(--ink3);font-size:13px;padding:10px 0}
.tablewrap{overflow-x:auto}
table{border-collapse:collapse;width:100%;font-size:13px}
th,td{text-align:right;padding:7px 10px;border-top:1px solid var(--grid);white-space:nowrap}
th{color:var(--ink2);font-weight:600;border-top:0}th:first-child,td:first-child{text-align:left}
footer{color:var(--ink3);font-size:12px;margin-top:14px;max-width:110ch}
</style>
</head>
<body>
<div class="wrap">
  <header>
    <div>
      <h1 id="title"></h1>
      <div class="sub" id="subtitle"></div>
    </div>
    <div class="sets" id="sets" role="group" aria-label="Dataset"></div>
    <div class="replay" role="group" aria-label="Replay controls">
      <button id="play" type="button">Play</button>
      <input id="slider" type="range" min="1" value="1" aria-label="Discharge cycle">
      <span class="now" id="now"></span>
      <label>Speed <select id="speed"><option value="600">1×</option><option value="250" selected>2×</option><option value="90">5×</option></select></label>
    </div>
  </header>

  <section class="top">
    <div class="bench"><div class="cells" id="mods" aria-label="Cells on the bench"></div></div>
    <div class="summary" id="kpis" aria-label="Group summary"></div>
  </section>

  <section class="grid charts">
    <div class="card"><div class="chead"><h2 id="t-soh"></h2><div class="legend" id="l-soh"></div></div><div class="plot" id="c-soh"></div></div>
    <div class="card"><div class="chead"><h2 id="t-rul"></h2><div class="legend" id="l-rul"></div></div><div class="plot" id="c-rul"></div></div>
    <div class="card"><div class="chead"><h2 id="t-soc"></h2><div class="legend" id="l-soc"></div></div><div class="plot" id="c-soc"></div></div>
    <div class="card"><div class="chead"><h2>Alerts and maintenance log</h2><div class="legend" id="l-al"></div></div><ul class="alerts" id="alerts"></ul></div>
  </section>

  <div class="card"><div class="chead"><h2 id="t-table"></h2></div><div class="tablewrap"><table id="table"></table></div></div>
  <footer id="foot"></footer>
</div>

<script>
const DATA = /*DATA*/null;
let M, META, U, ALERTS, S = 0;                              // the dataset being shown
const css = v => getComputedStyle(document.documentElement).getPropertyValue(v).trim();
const STATUS = {good:{label:"Healthy",icon:"✓",c:"--good"}, warning:{label:"Watch",icon:"▲",c:"--warning"},
                serious:{label:"Warning",icon:"◆",c:"--serious"}, critical:{label:"Critical",icon:"✕",c:"--critical"}, ended:{label:"Test ended",icon:"■",c:"--ink3"}};
let T = 1, sel = 0, timer = null;
const $ = id => document.getElementById(id);
const pct = (v, d = 1) => v == null ? "–" : (100 * v).toFixed(d) + "%";
const num = (v, d = 0) => v == null ? "–" : v.toFixed(d);
const idx = m => Math.min(T, m.n) - 1;                       // module index at replay time T
const pill = s => `<span class="pill"><i style="color:var(${STATUS[s].c})">${STATUS[s].icon}</i>${STATUS[s].label}</span>`;

function status(m, i) {
  if (m.sohEst[i] <= META.eolSoh) return "critical";
  if (m.rulPred[i] != null && m.rulPred[i] <= META.rulWarn) return "serious";
  if (m.sohEst[i] < META.sohWatch) return "warning";
  return "good";
}

function render() {
  const NOW = META.grid[T - 1];                              // the cycle number at replay step T
  $("now").textContent = `Cycle ${NOW} of ${META.maxK}`;
  $("slider").value = T;
  const live = M.map(m => ({m, i: idx(m), ended: T > m.n}));
  const on = live.some(x => !x.ended) ? live.filter(x => !x.ended) : live;        // cells still on test drive the summary
  const meanSoh = on.reduce((a, x) => a + x.m.sohEst[x.i], 0) / on.length;
  const cap = on.reduce((a, x) => a + x.m.sohEst[x.i] * META.rated, 0);
  const weak = on.reduce((a, x) => x.m.sohEst[x.i] < a.m.sohEst[a.i] ? x : a);
  const withRul = on.filter(x => x.m.rulPred[x.i] != null);
  const soon = withRul.length ? withRul.reduce((a, x) => x.m.rulPred[x.i] < a.m.rulPred[a.i] ? x : a) : null;
  const scope = on.length < live.length ? `${on.length} ${U}s still on test` : `all ${live.length} ${U}s`;
  const seen = ALERTS.filter(a => a.k <= NOW);
  $("kpis").innerHTML = [
    ["Group health (SOH)", pct(meanSoh), `Mean estimate, ${scope}`],
    ["Usable capacity", `${cap.toFixed(2)} <small>of ${(on.length * META.rated).toFixed(1)} Ah</small>`, `Estimated capacity, ${scope}`],
    [`Weakest ${U}`, `${weak.m.name} <small>${pct(weak.m.sohEst[weak.i])}</small>`, "Lowest estimated SOH"],
    ["Shortest remaining life", soon ? `${num(soon.m.rulPred[soon.i])} <small>cycles, ${soon.m.name}</small>` : "–",
       soon ? `90% band ${num(soon.m.rulLo[soon.i])} to ${num(soon.m.rulHi[soon.i])} cycles` : `Prediction starts at cycle ${META.firstRul}`],
    ["Alerts so far", String(seen.length), seen.length ? `Latest: ${seen[seen.length - 1].module}, cycle ${seen[seen.length - 1].k}` : "None raised"],
  ].map(([l, v, n]) => `<div class="stat"><div class="lab">${l}</div><div class="val">${v}</div><div class="note">${n}</div></div>`).join("");

  $("mods").innerHTML = live.map((x, j) => {
    const {m, i} = x, st = x.ended ? "ended" : status(m, i);
    return `<button class="cellbtn" type="button" aria-pressed="${j === sel}" data-j="${j}" aria-label="${m.name}, ${STATUS[st].label}, health ${pct(m.sohEst[i])}">
      ${battery(m.name, m.sohEst[i], st, j)}<div class="plate"></div>
      <div class="cname">${m.name}</div><div style="margin-top:4px">${pill(st)}</div>
      <div class="big">${pct(m.sohEst[i])}<small>health</small></div>
      <div class="line">${m.rulPred[i] == null ? "Life prediction starts at cycle " + META.firstRul : `<b>${num(m.rulPred[i])}</b> cycles left (${num(m.rulLo[i])} to ${num(m.rulHi[i])})`}</div>
      <div class="line">Peak <b>${num(m.tmax[i], 1)} °C</b>, cycle <b>${m.k[i]}</b>${x.ended ? " (last)" : ""}</div></button>`;
  }).join("");
  document.querySelectorAll(".cellbtn").forEach(b => b.onclick = () => { sel = +b.dataset.j; render(); });

  const m = M[sel], i = idx(m), upto = a => a.slice(0, i + 1), pts = a => upto(a).map((y, n) => ({x: m.k[n], y}));
  const ref = css("--ink2"), s1 = css("--s1"), s2 = css("--s2");
  $("t-soh").textContent = `State of health, ${m.name}`;
  legend("l-soh", [["Twin estimate", s1], ["Measured (reference)", ref]]);
  chart("c-soh", {xMax: META.maxK, xLabel: META.xLabel, unit: "%", scale: 100, dec: 1,
    series: [{name: "Measured (reference)", color: ref, w: 1.5, pts: pts(m.soh)}, {name: "Twin estimate", color: s1, w: 2, pts: pts(m.sohEst)}],
    hline: {y: META.eolSoh, label: "End of life"}, yPad: [META.eolSoh - .03, null]});
  $("t-rul").textContent = `Remaining useful life, ${m.name}`;
  legend("l-rul", [["Twin prediction", s1], ["90% band", s1, true], ["Actual (known afterwards)", ref]]);
  chart("c-rul", {xMax: META.maxK, xLabel: META.xLabel, unit: " cycles", scale: 1, dec: 0, yMin: 0,
    series: [{name: "Actual", color: ref, w: 1.5, pts: pts(m.rul)}, {name: "Twin prediction", color: s1, w: 2, pts: pts(m.rulPred)}],
    band: {color: s1, lo: pts(m.rulLo), hi: pts(m.rulHi)}});
  const sc = m.soc ? m.soc[i] : null;
  $("t-soc").textContent = `State of charge during discharge, ${m.name}, cycle ${m.k[i]}`;
  if (m.soc) legend("l-soc", [["Twin (ageing-aware)", s1], ["Fixed-capacity counting", s2], ["Reference", ref]]); else $("l-soc").innerHTML = "";
  if (sc) chart("c-soc", {xMax: Math.max(...sc.t), xLabel: "Time in discharge (min)", unit: "%", scale: 100, dec: 1, yMin: 0, yMax: 1,
    series: [{name: "Reference", color: ref, w: 1.5, pts: sc.t.map((x, n) => ({x, y: sc.ref[n]}))},
             {name: "Fixed-capacity counting", color: s2, w: 2, pts: sc.t.map((x, n) => ({x, y: sc.fixed[n]}))},
             {name: "Twin (ageing-aware)", color: s1, w: 2, pts: sc.t.map((x, n) => ({x, y: sc.twin[n]}))}]});
  else $("c-soc").innerHTML = `<div class="empty">${m.soc ? "No discharge curve recorded for this cycle." : META.socNote}</div>`;

  $("l-al").innerHTML = `<span>${seen.length} raised up to cycle ${NOW}</span>`;
  $("alerts").innerHTML = seen.length ? seen.slice().reverse().map(a =>
    `<li><span class="when">Cycle ${a.k}</span>${pill(a.level)}<span><b>${a.module}</b>: ${a.msg}</span></li>`).join("")
    : `<li class="empty" style="display:block">No alerts yet.</li>`;

  $("table").innerHTML = `<tr><th>${U[0].toUpperCase() + U.slice(1)}</th><th>Status</th><th>Cycle</th><th>SOH, twin</th><th>SOH, measured</th><th>RUL, twin</th><th>90% band</th><th>RUL, actual</th><th>Peak temp.</th></tr>` +
    live.map(x => { const {m, i} = x; return `<tr><td>${m.name}</td><td>${pill(x.ended ? "ended" : status(m, i))}</td><td>${m.k[i]}</td><td>${pct(m.sohEst[i])}</td><td>${pct(m.soh[i])}</td>
      <td>${num(m.rulPred[i])}</td><td>${m.rulPred[i] == null ? "–" : num(m.rulLo[i]) + " to " + num(m.rulHi[i])}</td><td>${num(m.rul[i])}</td><td>${num(m.tmax[i], 1)} °C</td></tr>`; }).join("");
}

const NUMFONT = "Bahnschrift,'DIN Alternate','Roboto Condensed','Arial Narrow',sans-serif";
const STEEL = j => `<linearGradient id="st${j}" x1="0" x2="1"><stop offset="0" stop-color="#98a2ad"/><stop offset=".3" stop-color="#f1f4f7"/><stop offset=".6" stop-color="#c3cbd3"/><stop offset="1" stop-color="#8b95a0"/></linearGradient>`;
const SHADE = (id, c) => `<linearGradient id="${id}" x1="0" x2="1"><stop offset="0" stop-color="#000" stop-opacity=".22"/><stop offset=".3" stop-color="#fff" stop-opacity=".22"/><stop offset=".6" stop-color="#000" stop-opacity="0"/><stop offset="1" stop-color="#000" stop-opacity=".25"/></linearGradient>`;

function battery(name, f, st, j) {
  const lv = Math.max(0, Math.min(1, f)), c = `var(${STATUS[st].c})`, style = META.style || "bars";
  if (style === "bars") return batteryBars(lv, c, j);
  if (style === "module") return batteryModule(name, lv, c, j);
  return batteryGlass(lv, c, j);
}

function batteryGlass(lv, c, j) {
  // a clear cylindrical cell: the coloured content is the estimated health, the dark line is the end-of-life level
  const x = 20, y0 = 30, w = 72, h = 168, y = y0 + h * (1 - lv), eol = y0 + h * (1 - META.eolSoh);
  return `<svg viewBox="0 0 112 214" aria-hidden="true"><defs>${STEEL(j)}${SHADE("sh" + j)}
      <clipPath id="bd${j}"><rect x="${x}" y="${y0}" width="${w}" height="${h}" rx="16"/></clipPath></defs>
    <ellipse cx="56" cy="203" rx="40" ry="6" fill="#17212b" opacity=".10"/>
    <rect x="41" y="8" width="30" height="14" rx="5" fill="url(#st${j})"/><rect x="26" y="19" width="60" height="14" rx="6" fill="url(#st${j})"/>
    <rect x="${x}" y="${y0}" width="${w}" height="${h}" rx="16" fill="#f3f6f9"/>
    <g clip-path="url(#bd${j})"><rect x="${x}" y="${y}" width="${w}" height="${y0 + h - y}" fill="${c}"/>
      <ellipse cx="56" cy="${y}" rx="${w / 2}" ry="6" fill="#fff" opacity=".35"/>
      <rect x="${x}" y="${y0}" width="${w}" height="${h}" fill="url(#sh${j})"/>
      <rect x="${x + 9}" y="${y0 + 8}" width="7" height="${h - 16}" rx="3.5" fill="#fff" opacity=".45"/></g>
    <rect x="${x}" y="${y0}" width="${w}" height="${h}" rx="16" fill="none" stroke="#aeb7c1" stroke-width="2"/>
    <line x1="${x - 6}" x2="${x + w + 6}" y1="${eol}" y2="${eol}" stroke="#17212b" stroke-width="2" stroke-dasharray="5 4"/>
    <text x="56" y="${Math.min(y0 + h - 14, Math.max(y + 30, y0 + 34))}" text-anchor="middle" fill="#fff" stroke="#17212b" stroke-opacity=".55" stroke-width="3" paint-order="stroke" style="font:600 21px ${NUMFONT}">${(100 * lv).toFixed(0)}%</text></svg>`;
}

function batteryBars(lv, c, j) {
  // the familiar battery symbol: five bars that empty from the top as health falls
  const x = 22, y0 = 30, w = 68, h = 166, pad = 9, n = 5, gap = 6, bh = (h - 2 * pad - gap * (n - 1)) / n, y = y0 + pad + (h - 2 * pad) * (1 - lv), eol = y0 + pad + (h - 2 * pad) * (1 - META.eolSoh);
  let slots = "", clip = "";
  for (let s = 0; s < n; s++) { const r = `x="${x + pad}" y="${y0 + pad + s * (bh + gap)}" width="${w - 2 * pad}" height="${bh}" rx="6"`; slots += `<rect ${r} fill="#e9edf1"/>`; clip += `<rect ${r}/>`; }
  return `<svg viewBox="0 0 112 214" aria-hidden="true"><defs><clipPath id="bs${j}">${clip}</clipPath></defs>
    <ellipse cx="56" cy="203" rx="38" ry="5" fill="#17212b" opacity=".08"/>
    <rect x="42" y="12" width="28" height="18" rx="6" fill="#17212b"/>
    <rect x="${x}" y="${y0}" width="${w}" height="${h}" rx="16" fill="#fff" stroke="#17212b" stroke-width="5"/>
    ${slots}<g clip-path="url(#bs${j})"><rect x="${x}" y="${y}" width="${w}" height="${y0 + h - y}" fill="${c}"/></g>
    <path d="M${x - 14} ${eol - 6}l9 6l-9 6z" fill="#17212b"/><path d="M${x + w + 14} ${eol - 6}l-9 6l9 6z" fill="#17212b"/></svg>`;
}

function batteryModule(name, lv, c, j) {
  // a storage-system style module: a box with two terminals, a ribbed case and a level window on the front
  const gx = 30, gy = 64, gw = 52, gh = 104, y = gy + gh * (1 - lv), eol = gy + gh * (1 - META.eolSoh);
  let ribs = ""; for (let r = 0; r < 4; r++) ribs += `<rect x="18" y="${178 + r * 5}" width="76" height="2" fill="#fff" opacity=".12"/>`;
  return `<svg viewBox="0 0 112 214" aria-hidden="true"><defs>${STEEL(j)}
      <linearGradient id="cs${j}" x1="0" x2="1"><stop offset="0" stop-color="#22344a"/><stop offset=".5" stop-color="#2f4864"/><stop offset="1" stop-color="#1a2838"/></linearGradient>
      <clipPath id="mw${j}"><rect x="${gx}" y="${gy}" width="${gw}" height="${gh}" rx="8"/></clipPath></defs>
    <ellipse cx="56" cy="204" rx="46" ry="5" fill="#17212b" opacity=".10"/>
    <rect x="24" y="14" width="18" height="16" rx="3" fill="url(#st${j})"/><rect x="70" y="14" width="18" height="16" rx="3" fill="url(#st${j})"/>
    <text x="33" y="11" text-anchor="middle" fill="#4d5966" style="font:700 12px system-ui,sans-serif">+</text><text x="79" y="11" text-anchor="middle" fill="#4d5966" style="font:700 12px system-ui,sans-serif">−</text>
    <rect x="10" y="26" width="92" height="176" rx="10" fill="url(#cs${j})"/>
    <rect x="10" y="26" width="92" height="24" rx="10" fill="#fff" opacity=".10"/>
    <text x="56" y="43" text-anchor="middle" fill="#fff" style="font:600 13px ${NUMFONT};letter-spacing:1.2px">${name}</text>
    <circle cx="90" cy="38" r="4" fill="${c}" stroke="#fff" stroke-width="1.5"/>
    <g clip-path="url(#mw${j})"><rect x="${gx}" y="${gy}" width="${gw}" height="${gh}" fill="#0a1119"/>
      <rect x="${gx}" y="${y}" width="${gw}" height="${gy + gh - y}" fill="${c}"/><rect x="${gx}" y="${y}" width="${gw}" height="3" fill="#fff" opacity=".35"/>
      <rect x="${gx + 5}" y="${gy}" width="5" height="${gh}" fill="#fff" opacity=".14"/></g>
    <rect x="${gx}" y="${gy}" width="${gw}" height="${gh}" rx="8" fill="none" stroke="#fff" stroke-opacity=".4"/>
    <line x1="${gx - 8}" x2="${gx + gw + 8}" y1="${eol}" y2="${eol}" stroke="#fff" stroke-width="2"/>
    <text x="56" y="${gy + gh - 10}" text-anchor="middle" fill="#fff" stroke="#0a1119" stroke-opacity=".5" stroke-width="3" paint-order="stroke" style="font:600 19px ${NUMFONT}">${(100 * lv).toFixed(0)}%</text>${ribs}</svg>`;
}

function legend(id, items) {
  $(id).innerHTML = items.map(([n, c, band]) => `<span><span class="key${band ? " band" : ""}" style="${band ? "background:" + c : "border-color:" + c}"></span>${n}</span>`).join("");
}

function ticks(lo, hi, n) {
  const span = hi - lo || 1, step0 = Math.pow(10, Math.floor(Math.log10(span / n))), err = span / n / step0;
  const step = step0 * (err >= 7.5 ? 10 : err >= 3.5 ? 5 : err >= 1.5 ? 2 : 1), out = [];
  for (let v = Math.ceil(lo / step) * step; v <= hi + 1e-9; v += step) out.push(+v.toFixed(10));
  return out;
}

function chart(id, o) {
  const el = $(id), W = Math.max(300, el.clientWidth || 520), H = 230, L = 46, R = 12, Tp = 10, B = 34;
  const all = o.series.flatMap(s => s.pts).filter(p => p.y != null).map(p => p.y)
    .concat(o.band ? o.band.hi.concat(o.band.lo).filter(p => p.y != null).map(p => p.y) : [], o.hline ? [o.hline.y] : []);
  let y0 = o.yMin != null ? o.yMin : Math.min(...all), y1 = o.yMax != null ? o.yMax : Math.max(...all);
  if (o.yPad && o.yPad[0] != null) y0 = Math.min(y0, o.yPad[0]);
  if (!isFinite(y0) || !isFinite(y1)) { y0 = 0; y1 = 1; }
  if (o.yMax == null) y1 += (y1 - y0) * 0.06 || 1; if (o.yMin == null) y0 -= (y1 - y0) * 0.04;
  const X = x => L + (W - L - R) * x / (o.xMax || 1), Y = y => Tp + (H - Tp - B) * (1 - (y - y0) / (y1 - y0 || 1));
  const path = pts => { let d = "", pen = false; for (const p of pts) { if (p.y == null) { pen = false; continue; } d += (pen ? "L" : "M") + X(p.x).toFixed(1) + " " + Y(p.y).toFixed(1); pen = true; } return d; };
  let g = "";
  for (const t of ticks(y0 * o.scale, y1 * o.scale, 4)) { const y = Y(t / o.scale); g += `<line x1="${L}" x2="${W - R}" y1="${y}" y2="${y}" stroke="${css("--grid")}"/><text x="${L - 6}" y="${y + 4}" text-anchor="end">${t}</text>`; }
  for (const t of ticks(0, o.xMax, 6)) g += `<text x="${X(t)}" y="${H - B + 15}" text-anchor="middle">${t}</text>`;
  g += `<line x1="${L}" x2="${W - R}" y1="${H - B}" y2="${H - B}" stroke="${css("--axis")}"/><text x="${(L + W - R) / 2}" y="${H - 4}" text-anchor="middle">${o.xLabel}</text>`;
  if (o.hline) { const y = Y(o.hline.y); g += `<line x1="${L}" x2="${W - R}" y1="${y}" y2="${y}" stroke="${css("--critical")}" stroke-width="1"/><text x="${W - R - 2}" y="${y - 4}" text-anchor="end">${o.hline.label}</text>`; }
  if (o.band) { const hi = o.band.hi.filter(p => p.y != null), lo = o.band.lo.filter(p => p.y != null);
    if (hi.length > 1) g += `<path d="${path(hi)}L${lo.slice().reverse().map(p => X(p.x).toFixed(1) + " " + Y(p.y).toFixed(1)).join("L")}Z" fill="${o.band.color}" opacity=".13"/>`; }
  for (const s of o.series) { g += `<path d="${path(s.pts)}" fill="none" stroke="${s.color}" stroke-width="${s.w}" stroke-linejoin="round" stroke-linecap="round"/>`;
    const last = s.pts.filter(p => p.y != null).pop(); if (last && s.w >= 2) g += `<circle cx="${X(last.x)}" cy="${Y(last.y)}" r="4" fill="${s.color}" stroke="${css("--surface")}" stroke-width="2"/>`; }
  el.innerHTML = `<svg viewBox="0 0 ${W} ${H}" role="img" aria-label="${$(id.replace("c-", "t-")).textContent}">${g}<line class="cross" y1="${Tp}" y2="${H - B}" stroke="${css("--axis")}" style="display:none"/><rect x="${L}" y="${Tp}" width="${W - L - R}" height="${H - Tp - B}" fill="transparent"/></svg><div class="tip"></div>`;
  const svg = el.firstChild, tip = el.lastChild, cross = svg.querySelector(".cross"), base = o.series[o.series.length - 1].pts;
  svg.onmousemove = e => {
    const box = svg.getBoundingClientRect(), px = (e.clientX - box.left) * W / box.width;
    if (!base.length) return;
    let n = 0, best = 1e9; base.forEach((p, k) => { const d = Math.abs(X(p.x) - px); if (d < best) { best = d; n = k; } });
    const x = base[n].x; cross.setAttribute("x1", X(x)); cross.setAttribute("x2", X(x)); cross.style.display = "";
    const rows = o.series.slice().reverse().map(s => { const p = s.pts.find(q => q.x === x); return p && p.y != null ? `<div><span><span class="key" style="border-color:${s.color}"></span> ${s.name}</span><b>${(p.y * o.scale).toFixed(o.dec)}${o.unit}</b></div>` : ""; }).join("");
    const bl = o.band && o.band.lo.find(q => q.x === x), bh = o.band && o.band.hi.find(q => q.x === x);
    tip.innerHTML = `<div style="color:var(--ink3)">${o.xLabel} ${Number.isInteger(x) ? x : x.toFixed(1)}</div>${rows}` + (bl && bl.y != null ? `<div><span>90% band</span><b>${bl.y.toFixed(0)} to ${bh.y.toFixed(0)}</b></div>` : "");
    tip.style.display = "block";
    const left = X(x) * box.width / W; tip.style.left = Math.min(left + 12, box.width - tip.offsetWidth - 4) + "px"; tip.style.top = "8px";
  };
  svg.onmouseleave = () => { tip.style.display = "none"; cross.style.display = "none"; };
}

function step(to) { T = Math.max(1, Math.min(META.grid.length, to)); render(); if (T >= META.grid.length) stop(); }
function stop() { clearInterval(timer); timer = null; $("play").textContent = "Play"; }
$("play").onclick = () => { if (timer) return stop(); if (T >= META.grid.length) T = 1; $("play").textContent = "Pause"; timer = setInterval(() => step(T + 1), +$("speed").value); };
$("speed").onchange = () => { if (timer) { stop(); $("play").click(); } };
$("slider").oninput = e => { stop(); step(+e.target.value); };
function use(s) {
  // switch dataset: every number on the page comes from the chosen set
  stop(); S = s; M = DATA.sets[s].modules; META = DATA.sets[s].meta; ALERTS = DATA.sets[s].alerts; U = META.unit; sel = 0;
  $("slider").max = META.grid.length; T = META.startT || 1;
  $("title").textContent = document.title = META.title; $("t-table").textContent = `Latest readings, all ${U}s`;
  $("subtitle").textContent = META.subtitle; $("foot").textContent = META.footnote;
  $("sets").innerHTML = DATA.sets.length < 2 ? "" : DATA.sets.map((d, n) => `<button type="button" aria-pressed="${n === s}" data-s="${n}">${d.meta.label}</button>`).join("");
  document.querySelectorAll("#sets button").forEach(b => b.onclick = () => use(+b.dataset.s));
  render();
}
addEventListener("resize", render);
use(0);
</script>
</body>
</html>
'''

import json, html as html_lib
DASH_UNIT = "cell"                           # what one battery is called on the page: "cell" now, "module" for a system built from modules
DASH_BATTERY_STYLE = "bars"                  # battery drawing: "glass", "bars" or "module"
RUL_METHOD = "Tuned XGBoost (frozen)"              # the best calibrated RUL method on the NASA cells (Part B)
SOH_WATCH, RUL_WARN, TEMP_LIMIT_C, N_SOC_POINTS = 0.80, 20, 45.0, 24          # NASA view: illustrative alert settings
LFP_SOH_WATCH, LFP_RUL_WARN, LFP_CELLS_SHOWN = 0.85, 100, 4                   # LFP view: illustrative alert settings

clean = lambda a, d=3: [None if v is None or not np.isfinite(v) else round(float(v), d) for v in a]

def make_alerts(name, k, est, pred, tmax, eol, watch, warn):
    # the first cycle at which each rule fires for one cell
    k, out = np.asarray(k), []
    first = lambda mask: int(k[np.asarray(mask)][0]) if np.any(mask) else None
    for level, at, msg in [("warning", first(np.asarray(est) < watch), f"SOH estimate below {watch:.0%}. Plan replacement."),
                           ("serious", first(np.nan_to_num(np.asarray(pred, float), nan=1e9) <= warn), f"Predicted remaining life at or below {warn} cycles. Schedule maintenance."),
                           ("critical", first(np.asarray(est) <= eol), f"SOH estimate at the end-of-life level ({eol:.1%}). Replace {DASH_UNIT}."),
                           ("serious", first(np.asarray(tmax) > TEMP_LIMIT_C), f"Peak temperature above {TEMP_LIMIT_C:.0f} °C.")]:
        if at is not None:
            out.append(dict(k=at, module=name, level=level, msg=msg))
    return out

# ---------------- view 1: the four NASA cells ----------------
soh_twin = soc_test.groupby(["Battery", "k"])["SOH_est"].first()
modules, alerts = [], []
for name in BATTERIES:
    b = df[df["Battery"] == name].reset_index(drop=True)
    stream = STREAM[name].set_index("k")
    est = b["k"].map(soh_twin[name]).interpolate(limit_direction="both")
    pred, lo, hi = (b["k"].map(stream[c]) for c in (RUL_METHOD, RUL_METHOD + " | low", RUL_METHOD + " | high"))
    curves, by_cycle = [], dict(tuple(soc_test[soc_test["Battery"] == name].groupby("k")))
    for k in b["k"]:
        d = by_cycle.get(k)
        if d is None:
            curves.append(None); continue
        d = d.iloc[np.unique(np.linspace(0, len(d) - 1, N_SOC_POINTS).astype(int))]
        curves.append(dict(t=clean(d["Time_s"] / 60, 2), ref=clean(d["SOC"]), twin=clean(d[SOC_METHODS[1]]), fixed=clean(d[SOC_METHODS[0]])))
    modules.append(dict(name=name, n=len(b), k=[int(v) for v in b["k"]], soh=clean(b["SOH"], 4), sohEst=clean(est, 4),
                        rul=clean(b["RUL"], 0), rulPred=clean(pred, 1), rulLo=clean(lo, 1), rulHi=clean(hi, 1),
                        tmax=clean(b["Temperature_max"], 1), soc=curves))
    alerts += make_alerts(name, b["k"], est, pred, b["Temperature_max"], SOH_EOL, SOH_WATCH, RUL_WARN)
    alerts += [dict(k=int(k), module=name, level="warning", msg="Abnormal charge detected (interrupted or partial). Charging features were filled from neighbouring cycles.")
               for k in b.loc[b["Charge_feature_filled"], "k"]]
n_max = max(m["n"] for m in modules)
sets = [dict(modules=modules, alerts=sorted(alerts, key=lambda a: a["k"]),
             meta=dict(label="NASA cells", title="Battery Digital Twin Monitor", unit=DASH_UNIT, style=DASH_BATTERY_STYLE, xLabel="Discharge cycle",
                       eolSoh=round(SOH_EOL, 4), rated=RATED_CAPACITY_AH, sohWatch=SOH_WATCH, rulWarn=RUL_WARN, firstRul=SEQ_LEN, startT=SEQ_LEN,
                       grid=list(range(1, n_max + 1)), maxK=n_max, socNote="",
                       subtitle="Replay of recorded NASA battery ageing tests. The four cylindrical cells are shown side by side as one small battery group. "
                                "Digital-shadow level: data flows from battery to twin, and no control signal is sent back.",
                       footnote=f"SOH estimate: linear model on four charging features. RUL: {RUL_METHOD.replace(' (frozen)', '')} with a 90% band calibrated on the training cells. "
                                f"SOC: charge counting corrected by the SOH estimate. Each {DASH_UNIT} is predicted by models trained on the other three. "
                                f"Measured SOH and actual RUL are shown for comparison and would not be available in live operation. "
                                f"Alert thresholds (SOH {SOH_WATCH:.0%}, RUL {RUL_WARN} cycles, {TEMP_LIMIT_C:.0f} °C) are illustrative settings, not validated maintenance rules. "
                                f"Data: NASA Ames Prognostics Data Repository, battery data set (B0005, B0006, B0007, B0018)."))]

# ---------------- view 2: LFP cells from the second dataset (needs Parts C, D and E) ----------------
if all(v in globals() for v in ("E", "PRED2", "G2", "K2", "Y2", "matr")):
    lfp_eol = EOL_AH / NOMINAL_AH
    order = np.argsort(life)                                             # cells from shortest to longest life
    shown = [int(order[int(q * (len(order) - 1))]) for q in np.linspace(0.1, 0.9, LFP_CELLS_SHOWN)]
    tmax_all = matr.set_index(["Battery", "k"])["Tmax"]
    modules2, alerts2 = [], []
    for g in shown:
        m, name = G2 == g, CELL_NAMES2[g]
        k, est, pred, half = K2[m], PRED2["Tuned XGBoost"][m, 0], E["xgb"][m], E["xgb_half"][m]
        tmax = tmax_all[name].reindex(k).interpolate(limit_direction="both").values
        modules2.append(dict(name=name, n=int(m.sum()), k=[int(v) for v in k], soh=clean(Y2[m, 0], 4), sohEst=clean(est, 4),
                             rul=clean(Y2[m, 1], 0), rulPred=clean(pred, 1), rulLo=clean(np.clip(pred - half, 0, None), 1), rulHi=clean(pred + half, 1),
                             tmax=clean(tmax, 1), soc=None))
        alerts2 += make_alerts(name, k, est, pred, tmax, lfp_eol, LFP_SOH_WATCH, LFP_RUL_WARN)
    longest = max(modules2, key=lambda mm: mm["n"])
    sets.append(dict(modules=modules2, alerts=sorted(alerts2, key=lambda a: a["k"]),
                     meta=dict(label="LFP cells", title="Battery Digital Twin Monitor", unit=DASH_UNIT, style=DASH_BATTERY_STYLE, xLabel="Cycle",
                               eolSoh=round(lfp_eol, 4), rated=NOMINAL_AH, sohWatch=LFP_SOH_WATCH, rulWarn=LFP_RUL_WARN, firstRul=int(longest["k"][0]), startT=1,
                               grid=longest["k"], maxK=int(longest["k"][-1]),
                               socNote="SOC estimation was studied on the NASA cells only. Switch to the NASA view to see it.",
                               subtitle=f"Replay of {LFP_CELLS_SHOWN} of the {len(CELL_NAMES2)} LFP cells in the MIT/Stanford/Toyota dataset, chosen from short-lived to long-lived. "
                                        "Digital-shadow level: data flows from battery to twin, and no control signal is sent back.",
                               footnote=f"SOH estimate and RUL: Tuned XGBoost on five charging-side features over a 20-cycle window; RUL has a 90% band calibrated on held-out training cells. "
                                        f"Each {DASH_UNIT} is predicted by models that never saw it (5-fold, grouped by cell). Every third cycle is shown. "
                                        f"Measured SOH and actual RUL are shown for comparison and would not be available in live operation. "
                                        f"Alert thresholds (SOH {LFP_SOH_WATCH:.0%}, RUL {LFP_RUL_WARN} cycles, {TEMP_LIMIT_C:.0f} °C) are illustrative settings, not validated maintenance rules. "
                                        f"Data: Severson et al., Nature Energy, 2019.")))
else:
    print("Second dataset results not in memory (Parts C, D and E), so the dashboard shows the NASA view only.")

page = DASHBOARD_TEMPLATE.replace("/*DATA*/null", json.dumps(dict(sets=sets)))
dash_path = os.path.join(OUT, "twin_dashboard.html")
with open(dash_path, "w", encoding="utf-8") as fh:
    fh.write(page)
print(f"Dashboard saved: {dash_path}  ({len(page) / 1e6:.2f} MB, views: {[s['meta']['label'] for s in sets]})")
print("Open that file in any web browser; it needs no internet and no Python.")
try:
    from IPython.display import HTML, display
    display(HTML(f'<iframe srcdoc="{html_lib.escape(page)}" style="width:100%;height:1650px;border:0"></iframe>'))
except ImportError:
    pass

## G2. Export high-resolution figures of the dashboard
Optional. Saves six images (full page, cell panel and charts, for each view) to `stage2_results/dashboard_figures`, about 3,800 pixels wide. The first cell installs a browser tool and takes 1 to 2 minutes. Change `SHOTS` to choose a different cycle or cell.

In [ ]:
!pip -q install playwright
!playwright install --with-deps chromium

In [ ]:
import pathlib
from playwright.async_api import async_playwright

FIG_DIR = os.path.join(OUT, "dashboard_figures"); os.makedirs(FIG_DIR, exist_ok=True)
# (file name, view: 0 = NASA cells, 1 = LFP cells, cycle to show, cell to select: 0 = first)
SHOTS = [("dashboard_nasa", 0, 84, 1), ("dashboard_lfp", 1, 500, 0)]   # the NASA pictures in the paper: cycle 84, cell B0006

async def export_dashboard(scale=3):
    async with async_playwright() as p:
        browser = await p.chromium.launch(args=["--no-sandbox"])
        page = await browser.new_page(viewport={"width": 1280, "height": 900}, device_scale_factor=scale)
        await page.goto(pathlib.Path(os.path.join(OUT, "twin_dashboard.html")).as_uri())
        views = await page.evaluate("DATA.sets.length")
        for name, view, cycle, cell in SHOTS:
            if view >= views:
                print("skipped", name, "(that view is not in the dashboard file)"); continue
            await page.evaluate(f"use({view}); step(Math.max(1, META.grid.findIndex(k => k >= {cycle}) + 1))")
            await page.locator(".cellbtn").nth(cell).click()
            await page.screenshot(path=os.path.join(FIG_DIR, name + "_full.png"), full_page=True)
            await page.locator("section.top").screenshot(path=os.path.join(FIG_DIR, name + "_cells.png"))
            await page.locator("section.charts").screenshot(path=os.path.join(FIG_DIR, name + "_charts.png"))
            print("saved", name, "at", await page.evaluate("document.getElementById('now').textContent"))
        await browser.close()

await export_dashboard()
print("Figures are in:", FIG_DIR)

# Part H — Final tables for the paper
Collects every result into paper-ready tables, as mean ± standard deviation, and saves each as a CSV file and a LaTeX file in `stage2_results/paper_tables`.

- NASA tables: the spread is across the four test cells.
- LFP tables: the spread is across the five folds.

It reads the saved results, so it takes a few seconds. Run it after all the parts above.

## H1. Build and save the tables

In [ ]:
TAB_DIR = os.path.join(OUT, "paper_tables"); os.makedirs(TAB_DIR, exist_ok=True)
pm = lambda s, d: f"{s.mean():.{d}f} ± {s.std(ddof=1):.{d}f}"          # mean ± sample standard deviation

def save_table(name, table, caption):
    table.to_csv(os.path.join(TAB_DIR, name + ".csv"))
    esc = lambda v: str(v).replace("±", r"$\pm$").replace("%", r"\%").replace("_", r"\_").replace("R2", r"$R^2$")
    head = [table.index.name or ""] + [esc(c) for c in table.columns]
    rows = [" & ".join([esc(i)] + [esc(v) for v in r]) + r" \\" for i, r in zip(table.index, table.values)]
    tex = "\n".join([r"\begin{table}[t]", r"\centering", r"\caption{" + caption.replace("±", r"$\pm$") + "}", r"\label{tab:" + name + "}",
                     r"\begin{tabular}{l" + "c" * len(table.columns) + "}", r"\toprule", " & ".join(head) + r" \\", r"\midrule",
                     *rows, r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    with open(os.path.join(TAB_DIR, name + ".tex"), "w", encoding="utf-8") as fh:
        fh.write(tex + "\n")
    print(f"\n{caption}"); print(table)

# ---- Table 1: NASA cells, five models, two input sets (leave-one-battery-out) ----
a = pd.DataFrame(load('A6_v3')[0])
a["Method"] = a["Inputs"] + " | " + a["Model"]
t1 = a.groupby("Method", sort=False).apply(lambda g: pd.Series({"SOH RMSE (%)": pm(100 * g["SOH_RMSE"], 2), "SOH R2": pm(g["SOH_R2"], 3),
                                                                "RUL RMSE (cycles)": pm(g["RUL_RMSE"], 1), "RUL R2": pm(g["RUL_R2"], 2)}))
save_table("nasa_model_comparison", t1, "NASA cells: SOH and RUL by input set and model, leave-one-battery-out (mean ± s.d. over four test cells).")

# ---- Table 2: NASA cells, RUL with confidence bands (frozen models and the self-updating attempts) ----
b = pd.DataFrame(load('B2_v4')[0])
t2 = b.groupby("Method", sort=False).apply(lambda g: pd.Series({"RUL RMSE (cycles)": pm(g["RUL_RMSE"], 1), "RUL R2": pm(g["RUL_R2"], 2),
                                                                "90% band coverage": pm(g["Coverage_90"], 2), "Band width (cycles)": pm(g["Band_width"], 0)}))
save_table("nasa_rul_uncertainty", t2, "NASA cells: RUL accuracy and 90\\% band coverage (mean ± s.d. over four test cells).")

# ---- Table 3: LFP cells, five models (5-fold, grouped by cell) ----
folds2 = [te for _, te in GroupKFold(n_splits=5).split(SEQ2, groups=G2)]            # the same folds as Parts D and E
def per_fold(p_soh, p_rul):
    out = []
    for te in folds2:
        pre = PRE2[te]; row = {}
        if p_soh is not None:
            row["SOH RMSE (%)"] = 100 * np.sqrt(mean_squared_error(Y2[te, 0], p_soh[te])); row["SOH R2"] = r2_score(Y2[te, 0], p_soh[te])
        row["RUL RMSE (cycles)"] = np.sqrt(mean_squared_error(Y2[te, 1][pre], p_rul[te][pre]))
        row["RUL MAE (cycles)"] = mean_absolute_error(Y2[te, 1][pre], p_rul[te][pre]); row["RUL R2"] = r2_score(Y2[te, 1][pre], p_rul[te][pre])
        out.append(row)
    return pd.DataFrame(out)
DEC = {"SOH RMSE (%)": 2, "SOH R2": 3, "RUL RMSE (cycles)": 0, "RUL MAE (cycles)": 0, "RUL R2": 2, "90% band coverage": 2, "Band width (cycles)": 0}
fmt = lambda f: pd.Series({c: pm(f[c], DEC[c]) for c in f.columns})
t3 = pd.DataFrame({m: fmt(per_fold(PRED2[m][:, 0], np.clip(PRED2[m][:, 1], 0, None))) for m in MODELS2}).T
t3.index.name = "Model"
save_table("lfp_model_comparison", t3, "LFP cells: SOH and RUL by model, 5-fold cross-validation grouped by cell (mean ± s.d. over folds).")

# ---- Table 4: LFP cells, Tuned XGBoost, the twin filter and calibrated bands ----
def band_cols(p, half):
    f = per_fold(None, p)
    y = Y2[:, 1]; inside = (y >= np.clip(p - half, 0, None)) & (y <= p + half)
    f["90% band coverage"] = [inside[te][PRE2[te]].mean() for te in folds2]; f["Band width (cycles)"] = [(2 * half)[te][PRE2[te]].mean() for te in folds2]
    return fmt(f)
t4 = pd.DataFrame({"Tuned XGBoost": band_cols(E["xgb"], E["xgb_half"]), "Self-updating twin filter": band_cols(E["twin"], E["twin_half"])}).T
t4.index.name = "Method"
save_table("lfp_rul_uncertainty", t4, "LFP cells: RUL with calibrated 90\\% bands, cells unseen in training (mean ± s.d. over folds).")

# ---- Table 5: SOC on the NASA cells ----
s = load('F1_v2')
e2 = lambda d, m: 100 * np.sqrt(((d[m] - d["SOC"]) ** 2).mean())
frac_h = s["k"] / s.groupby("Battery")["k"].transform("max")
t5 = pd.DataFrame({m: {"All cycles": pm(s.groupby("Battery").apply(lambda d: e2(d, m)), 2),
                       "First third of life": pm(s[frac_h <= 1 / 3].groupby("Battery").apply(lambda d: e2(d, m)), 2),
                       "Last third of life": pm(s[frac_h > 2 / 3].groupby("Battery").apply(lambda d: e2(d, m)), 2)} for m in SOC_METHODS}).T
t5.index.name = "Method"
save_table("nasa_soc", t5, "NASA cells: SOC RMSE in percentage points, leave-one-battery-out (mean ± s.d. over four test cells).")

pd.DataFrame([dict(Setting="NASA: seeds per deep model", Value=N_SEEDS), dict(Setting="LFP: seeds per deep model", Value=DEEP_SEEDS),
              dict(Setting="LFP: every n-th cycle used", Value=STEP), dict(Setting="Window length (cycles)", Value=SEQ_LEN),
              dict(Setting="NASA end-of-life capacity (Ah)", Value=EOL_USED_AH), dict(Setting="LFP end-of-life capacity (Ah)", Value=EOL_AH),
              dict(Setting="LFP cells", Value=len(CELL_NAMES2))]).to_csv(os.path.join(TAB_DIR, "run_settings.csv"), index=False)
print("\nSaved CSV and LaTeX tables, plus run_settings.csv, to", TAB_DIR)

# Part I — Model cost, sensor faults, statistical tests and input importance
All on the LFP cells, using the windows and folds of Part D.

## I1 and I2. Model cost and sensor faults

In [ ]:
# ---------------- I1 + I2: one train/test split, all five models kept so they can be measured and stressed ----------------
import tempfile
NOISE_LEVELS   = [0.005, 0.01, 0.02, 0.05]     # each reading off by a random amount of this relative size
MISSING_LEVELS = [0.10, 0.30, 0.50]            # share of cycles lost; the last received value is held
GLITCH_LEVELS  = [0.02, 0.05, 0.10]            # share of readings replaced by a value 20% off
FAULT_REPEATS  = 3                             # random draws averaged for every fault setting
GLITCH_SIZE    = 0.20

def add_noise(S, level, rng):
    return S * (1 + level * rng.standard_normal(S.shape))
def add_missing(S, level, rng):
    S = S.copy(); lost = rng.random(S.shape[:2]) < level; lost[:, 0] = False
    for t in range(1, S.shape[1]):
        S[lost[:, t], t, :] = S[lost[:, t], t - 1, :]
    return S
def add_glitch(S, level, rng):
    hit = rng.random(S.shape) < level
    return np.where(hit, S * (1 + GLITCH_SIZE * rng.choice([-1.0, 1.0], size=S.shape)), S)
FAULTS = [("Noise", add_noise, NOISE_LEVELS), ("Missing cycles", add_missing, MISSING_LEVELS), ("Glitches", add_glitch, GLITCH_LEVELS)]

if have('I1_v1'):
    COST, FAULT = load('I1_v1')
else:
    tr, te = next(GroupKFold(n_splits=5).split(SEQ2, groups=G2))            # the first fold of Part D
    xs = StandardScaler().fit(SEQ2[tr].reshape(-1, nf)); ys = StandardScaler().fit(Y2[tr])
    scale = lambda S: xs.transform(S.reshape(-1, nf)).reshape(-1, SEQ_LEN, nf)
    Str = scale(SEQ2[tr]); ytr = ys.transform(Y2[tr])
    t0 = time.time()
    lin = Ridge(alpha=1.0).fit(Str[:, -1, :], ytr)
    rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(summary2(Str), ytr)
    xg = [new_xgb(**XGB_PARAMS2[0][t]).fit(W_raw[tr], Y2[tr, j]) for j, t in enumerate(["SOH", "RUL"])]
    set_seed(0)
    val_cells = np.random.choice(np.unique(G2[tr]), max(1, len(np.unique(G2[tr])) // 7), replace=False)
    va = np.isin(G2[tr], val_cells)
    deep = {}
    for kind in ["LSTM", "CNN-LSTM"]:
        set_seed(0); deep[kind] = fit_fast(build_model(nf, kind), Str[~va], ytr[~va], Str[va], ytr[va])
    print(f"models trained in {time.time() - t0:.0f} s")

    def deep_call(m):
        return (lambda X: np.asarray(m(X, training=False))) if hasattr(m, "count_params") else (lambda X: m.predict(X))
    PREDICT = {"Linear": lambda S: ys.inverse_transform(lin.predict(scale(S)[:, -1, :])),
               "Random Forest": lambda S: ys.inverse_transform(rf.predict(summary2(scale(S)))),
               "Tuned XGBoost": lambda S: np.column_stack([m.predict(summary2(S)) for m in xg]),
               "LSTM": lambda S: ys.inverse_transform(deep["LSTM"].predict(scale(S), verbose=0, batch_size=1024)),
               "CNN-LSTM": lambda S: ys.inverse_transform(deep["CNN-LSTM"].predict(scale(S), verbose=0, batch_size=1024))}
    ONE = {"Linear": PREDICT["Linear"], "Random Forest": PREDICT["Random Forest"], "Tuned XGBoost": PREDICT["Tuned XGBoost"],
           "LSTM": lambda S, f=deep_call(deep["LSTM"]): ys.inverse_transform(f(scale(S))),
           "CNN-LSTM": lambda S, f=deep_call(deep["CNN-LSTM"]): ys.inverse_transform(f(scale(S)))}

    def size_kb(obj):
        if hasattr(obj, "count_params"):                                   # Keras model: size of the saved file
            path = os.path.join(tempfile.mkdtemp(), "m.keras"); obj.save(path); return os.path.getsize(path) / 1024
        return len(pickle.dumps(obj)) / 1024
    OBJECTS = {"Linear": [lin], "Random Forest": [rf], "Tuned XGBoost": xg, "LSTM": [deep["LSTM"]], "CNN-LSTM": [deep["CNN-LSTM"]]}

    def score(P):
        pre = PRE2[te]; r = np.clip(P[:, 1], 0, None)
        return (100 * np.sqrt(mean_squared_error(Y2[te, 0], P[:, 0])), np.sqrt(mean_squared_error(Y2[te, 1][pre], r[pre])))

    rows = []
    one = SEQ2[te][:1]
    for mname in MODELS2:
        ONE[mname](one)                                                    # warm-up call
        times = []
        for _ in range(100):
            t1 = time.perf_counter(); ONE[mname](one); times.append(1000 * (time.perf_counter() - t1))
        t1 = time.perf_counter(); P = PREDICT[mname](SEQ2[te]); batch = 1000 * (time.perf_counter() - t1)
        soh, rul = score(P)
        rows.append(dict(Model=mname, Size_kB=sum(size_kb(o) for o in OBJECTS[mname]), One_prediction_ms=float(np.median(times)),
                         Per_1000_ms=1000 * batch / len(te), SOH_RMSE_pct=soh, RUL_RMSE=rul))
    COST = pd.DataFrame(rows).set_index("Model")

    rows = []
    for mname in MODELS2:
        soh, rul = score(PREDICT[mname](SEQ2[te]))
        rows.append(dict(Fault="Clean", Level=0.0, Model=mname, SOH_RMSE_pct=soh, RUL_RMSE=rul))
    for fname, fn, levels in FAULTS:
        for level in levels:
            draws = [fn(SEQ2[te], level, np.random.default_rng(100 + r)) for r in range(FAULT_REPEATS)]
            for mname in MODELS2:
                sc = np.mean([score(PREDICT[mname](S)) for S in draws], axis=0)
                rows.append(dict(Fault=fname, Level=level, Model=mname, SOH_RMSE_pct=sc[0], RUL_RMSE=sc[1]))
        print(f"{fname}: done")
    FAULT = pd.DataFrame(rows)
    save('I1_v1', (COST, FAULT))

t6 = pd.DataFrame({"Size (kB)": COST["Size_kB"].map("{:.0f}".format), "One prediction (ms)": COST["One_prediction_ms"].map("{:.2f}".format),
                   "1000 predictions (ms)": COST["Per_1000_ms"].map("{:.1f}".format),
                   "SOH RMSE (%)": COST["SOH_RMSE_pct"].map("{:.2f}".format), "RUL RMSE (cycles)": COST["RUL_RMSE"].map("{:.0f}".format)})
t6.index.name = "Model"
save_table("lfp_model_cost", t6, "LFP cells: model size, prediction time on a standard processor and accuracy on one held-out fold.")

FAULT["Condition"] = np.where(FAULT["Fault"] == "Clean", "Clean", FAULT["Fault"] + " " + (100 * FAULT["Level"]).map("{:g}".format) + "%")
order = list(dict.fromkeys(FAULT["Condition"]))
for col, name, cap, d in [("RUL_RMSE", "lfp_faults_rul", "RUL RMSE in cycles", 0), ("SOH_RMSE_pct", "lfp_faults_soh", "SOH RMSE in %", 2)]:
    t = FAULT.pivot(index="Condition", columns="Model", values=col).loc[order, MODELS2].apply(lambda c: c.map(lambda v: f"{v:.{d}f}"))
    save_table(name, t, f"LFP cells: {cap} when the health features are disturbed (one held-out fold, mean of {FAULT_REPEATS} random draws).")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for ax, (fname, _, levels) in zip(axes, FAULTS):
    for mname in MODELS2:
        d = FAULT[(FAULT["Model"] == mname) & FAULT["Fault"].isin([fname, "Clean"])].sort_values("Level")
        ax.plot(100 * d["Level"], d["RUL_RMSE"], marker="o", label=mname)
    ax.set_title(fname); ax.set_xlabel("Share or size of the disturbance (%)"); ax.grid(alpha=0.3)
axes[0].set_ylabel("RUL RMSE (cycles)"); axes[0].set_yscale("log"); axes[-1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT2, "matr_sensor_faults.png"), dpi=300); plt.show()

## I3. Statistical test

In [ ]:
# ---------------- I3: is the best model really better? paired test over the 124 cells ----------------
from scipy.stats import wilcoxon
cells_ = np.unique(G2)
def per_cell_rmse(mname, target):
    p = PRED2[mname].copy(); p[:, 1] = np.clip(p[:, 1], 0, None)
    j = 0 if target == "SOH" else 1
    keep = np.ones(len(G2), bool) if target == "SOH" else PRE2
    return np.array([np.sqrt(((Y2[(G2 == g) & keep, j] - p[(G2 == g) & keep, j]) ** 2).mean()) for g in cells_])

REF = "Tuned XGBoost"
rows = []
for target, unit, mult in [("SOH", "%", 100), ("RUL", "cycles", 1)]:
    ref = mult * per_cell_rmse(REF, target)
    others = [m for m in MODELS2 if m != REF]
    raw_p = [wilcoxon(ref, mult * per_cell_rmse(m, target)).pvalue for m in others]
    rank = np.argsort(raw_p); holm = np.empty(len(raw_p)); running = 0.0
    for pos, i in enumerate(rank):                                          # Holm correction for several comparisons
        running = max(running, min(1.0, (len(raw_p) - pos) * raw_p[i])); holm[i] = running
    rows.append(dict(Target=target, Model=REF, Median=np.median(ref), Wins=np.nan, p=np.nan, unit=unit))
    for m, p in zip(others, holm):
        e = mult * per_cell_rmse(m, target)
        rows.append(dict(Target=target, Model=m, Median=np.median(e), Wins=int((ref < e).sum()), p=p, unit=unit))
STAT = pd.DataFrame(rows)
fmt_p = lambda p: "reference" if np.isnan(p) else ("<0.001" if p < 0.001 else f"{p:.3f}")
t9 = pd.DataFrame({"Target": STAT["Target"] + " (" + STAT["unit"] + ")", "Model": STAT["Model"],
                   "Median RMSE per cell": [f"{v:.2f}" if t == "SOH" else f"{v:.0f}" for v, t in zip(STAT["Median"], STAT["Target"])],
                   f"Cells where {REF} is better (of {len(cells_)})": ["-" if np.isnan(w) else str(int(w)) for w in STAT["Wins"]],
                   "p-value (Wilcoxon, Holm)": STAT["p"].map(fmt_p)}).set_index("Target")
save_table("lfp_statistical_test", t9, f"LFP cells: per-cell error of every model against {REF} (paired Wilcoxon signed-rank test, Holm-corrected).")

## I4. Input importance and ablation

In [ ]:
# ---------------- I4: which inputs matter (permutation importance) and what is lost without them (ablation) ----------------
N_F = len(FEATURES2)
COLS_OF = {f: [i, N_F + i, 2 * N_F + i] for i, f in enumerate(FEATURES2)}          # latest value, window mean, window trend
PARTS = {"Latest value": list(range(N_F)), "Window mean": list(range(N_F, 2 * N_F)), "Window trend": list(range(2 * N_F, 3 * N_F))}
VARIANTS = {"All inputs": list(range(3 * N_F)), **{f"Without {f}": [c for c in range(3 * N_F) if c not in COLS_OF[f]] for f in FEATURES2},
            "Latest values only (no window)": PARTS["Latest value"]}
PERM_REPEATS = 3

if have('I4_v1'):
    IMP, ABL = load('I4_v1')
else:
    imp_rows, abl_rows = [], []
    for fold, (tr, te) in enumerate(GroupKFold(n_splits=5).split(SEQ2, groups=G2)):
        t0 = time.time(); pre = PRE2[te]
        def errors(ms, W):
            return (100 * np.sqrt(mean_squared_error(Y2[te, 0], ms[0].predict(W))),
                    np.sqrt(mean_squared_error(Y2[te, 1][pre], np.clip(ms[1].predict(W), 0, None)[pre])))
        fit = lambda cols: [new_xgb(**XGB_PARAMS2[fold][t]).fit(W_raw[tr][:, cols], Y2[tr, j]) for j, t in enumerate(["SOH", "RUL"])]
        full = fit(VARIANTS["All inputs"]); base = errors(full, W_raw[te])
        for group, sets in [("Input", COLS_OF), ("Window summary", PARTS)]:
            for name, cols in sets.items():
                inc = []
                for r in range(PERM_REPEATS):
                    W = W_raw[te].copy(); W[:, cols] = W[np.random.default_rng(10 * fold + r).permutation(len(W))][:, cols]
                    inc.append(np.subtract(errors(full, W), base))
                inc = np.mean(inc, axis=0)
                imp_rows.append(dict(Fold=fold + 1, Group=group, Name=name, SOH=inc[0], RUL=inc[1]))
        for name, cols in VARIANTS.items():
            e = base if name == "All inputs" else errors(fit(cols), W_raw[te][:, cols])
            abl_rows.append(dict(Fold=fold + 1, Variant=name, SOH=e[0], RUL=e[1]))
        print(f"fold {fold + 1}/5 done in {time.time() - t0:.0f} s")
    IMP, ABL = pd.DataFrame(imp_rows), pd.DataFrame(abl_rows)
    save('I4_v1', (IMP, ABL))

t10 = IMP.groupby(["Group", "Name"], sort=False).apply(lambda g: pd.Series({"SOH RMSE increase (% points)": pm(g["SOH"], 2),
                                                                           "RUL RMSE increase (cycles)": pm(g["RUL"], 0)}))
t10.index = [f"{g}: {n}" for g, n in t10.index]; t10.index.name = "Shuffled"
save_table("lfp_importance", t10, "LFP cells: rise in Tuned XGBoost error when an input is shuffled (mean ± s.d. over folds).")
t11 = ABL.groupby("Variant", sort=False).apply(lambda g: pd.Series({"SOH RMSE (%)": pm(g["SOH"], 2), "RUL RMSE (cycles)": pm(g["RUL"], 0)}))
save_table("lfp_ablation", t11, "LFP cells: Tuned XGBoost retrained with inputs removed (mean ± s.d. over folds).")

m = IMP[IMP["Group"] == "Input"].groupby("Name", sort=False)[["SOH", "RUL"]].agg(["mean", "std"])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, target, unit in zip(axes, ["SOH", "RUL"], ["% points", "cycles"]):
    ax.barh(m.index, m[(target, "mean")], xerr=m[(target, "std")], color="tab:blue", alpha=0.8)
    ax.invert_yaxis(); ax.set_xlabel(f"Rise in {target} RMSE when shuffled ({unit})"); ax.grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.savefig(os.path.join(OUT2, "matr_importance.png"), dpi=300); plt.show()
print("\nPart I finished. Tables are in", TAB_DIR)

## I5. Making Tuned XGBoost tolerant of disturbed readings

In [ ]:
# ---------------- I5: making Tuned XGBoost tolerant of disturbed readings ----------------
# I2 showed that the tree models lose accuracy quickly when readings are noisy. Two simple remedies are tested here:
#   Smoothed inputs : the window is summarised with medians of 5 cycles instead of single cycles
#   Noise training  : the model also sees disturbed copies of the training windows
AUG_NOISE = [0.01, 0.02]                       # noise levels added to the training copies
def robust_summary(S, n=5):
    first, last = np.median(S[:, :n, :], axis=1), np.median(S[:, -n:, :], axis=1)
    return np.concatenate([last, S.mean(axis=1), last - first], axis=1)
SUMMARIES = {"Standard": summary2, "Smoothed inputs": robust_summary, "Noise training": summary2, "Smoothed + noise training": robust_summary}

if have('I5_v1'):
    ROBUST = load('I5_v1')
else:
    tr, te = next(GroupKFold(n_splits=5).split(SEQ2, groups=G2)); pre = PRE2[te]
    rng = np.random.default_rng(7)
    S_aug = np.concatenate([SEQ2[tr]] + [add_noise(SEQ2[tr], a, rng) for a in AUG_NOISE])
    Y_aug = np.concatenate([Y2[tr]] * (1 + len(AUG_NOISE)))
    fitted = {}
    for vname, summ in SUMMARIES.items():
        t0 = time.time()
        S_fit, Y_fit = (S_aug, Y_aug) if "oise training" in vname else (SEQ2[tr], Y2[tr])
        fitted[vname] = [new_xgb(**XGB_PARAMS2[0][t]).fit(summ(S_fit), Y_fit[:, j]) for j, t in enumerate(["SOH", "RUL"])]
        print(f"{vname}: trained in {time.time() - t0:.0f} s")
    def score_v(vname, S):
        W = SUMMARIES[vname](S); ms = fitted[vname]
        return (100 * np.sqrt(mean_squared_error(Y2[te, 0], ms[0].predict(W))),
                np.sqrt(mean_squared_error(Y2[te, 1][pre], np.clip(ms[1].predict(W), 0, None)[pre])))
    rows = [dict(Fault="Clean", Level=0.0, Variant=v, SOH_RMSE_pct=score_v(v, SEQ2[te])[0], RUL_RMSE=score_v(v, SEQ2[te])[1]) for v in SUMMARIES]
    for fname, fn, levels in FAULTS:
        for level in levels:
            draws = [fn(SEQ2[te], level, np.random.default_rng(100 + r)) for r in range(FAULT_REPEATS)]      # the same draws as I2
            for v in SUMMARIES:
                sc = np.mean([score_v(v, S) for S in draws], axis=0)
                rows.append(dict(Fault=fname, Level=level, Variant=v, SOH_RMSE_pct=sc[0], RUL_RMSE=sc[1]))
    ROBUST = pd.DataFrame(rows)
    save('I5_v1', ROBUST)

ROBUST["Condition"] = np.where(ROBUST["Fault"] == "Clean", "Clean", ROBUST["Fault"] + " " + (100 * ROBUST["Level"]).map("{:g}".format) + "%")
order = list(dict.fromkeys(ROBUST["Condition"]))
for col, name, cap, d in [("RUL_RMSE", "lfp_robust_rul", "RUL RMSE in cycles", 0), ("SOH_RMSE_pct", "lfp_robust_soh", "SOH RMSE in %", 2)]:
    t = ROBUST.pivot(index="Condition", columns="Variant", values=col).loc[order, list(SUMMARIES)]
    t["LSTM (from I2)"] = FAULT[FAULT["Model"] == "LSTM"].set_index("Condition")[col].reindex(order)
    t = t.apply(lambda c: c.map(lambda v: f"{v:.{d}f}"))
    save_table(name, t, f"LFP cells: {cap} of four Tuned XGBoost variants under disturbed health features (one held-out fold).")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for ax, (fname, _, levels) in zip(axes, FAULTS):
    for v in SUMMARIES:
        d = ROBUST[(ROBUST["Variant"] == v) & ROBUST["Fault"].isin([fname, "Clean"])].sort_values("Level")
        ax.plot(100 * d["Level"], d["RUL_RMSE"], marker="o", label=v)
    d = FAULT[(FAULT["Model"] == "LSTM") & FAULT["Fault"].isin([fname, "Clean"])].sort_values("Level")
    ax.plot(100 * d["Level"], d["RUL_RMSE"], marker="s", ls="--", color="gray", label="LSTM")
    ax.set_title(fname); ax.set_xlabel("Share or size of the disturbance (%)"); ax.grid(alpha=0.3)
axes[0].set_ylabel("RUL RMSE (cycles)"); axes[-1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT2, "matr_robust_xgboost.png"), dpi=300); plt.show()

# Part J — Paper figures
Every figure is drawn at the printed width of the page (5.95 inches) with 8-point text. Each line has its own colour, marker shape and line style, so the figures stay readable in black and white. Saved as PDF and PNG in `stage2_results/paper_figures`.

In [ ]:
# ---------------- J1: all paper figures, drawn at the real printed size so the text stays readable ----------------
# The page's text area is 5.95 inches wide. Every figure is made exactly that wide with 8-point text, so nothing is
# shrunk when it is placed in the paper. Each line has its own colour, marker shape and line style (readable in black and white).
import math
from matplotlib.ticker import MaxNLocator
FIG_DIR = os.path.join(OUT, "paper_figures"); os.makedirs(FIG_DIR, exist_ok=True)
PAGE_W = 5.95
plt.rcParams.update({"font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8.5, "legend.fontsize": 8, "xtick.labelsize": 7.5,
                     "ytick.labelsize": 7.5, "axes.spines.top": False, "axes.spines.right": False, "pdf.fonttype": 42,
                     "axes.linewidth": 0.7, "xtick.major.width": 0.7, "ytick.major.width": 0.7})
LOOKS = [dict(color="#2a6fb0", marker="o", ls="-"), dict(color="#c9742a", marker="s", ls="--"), dict(color="#2f8f6b", marker="^", ls="-."),
         dict(color="#8d55a8", marker="D", ls=":"), dict(color="#a8483c", marker="v", ls=(0, (5, 1, 1, 1, 1, 1)))]
MODEL_LOOK = dict(zip(["Tuned XGBoost", "Random Forest", "LSTM", "CNN-LSTM", "Linear"], LOOKS))   # the same look in every figure
ACTUAL = dict(color="black", ls="-", lw=1.7)

def draw(ax, x, y, look, label, n_marks=6, slot=0, total=1):
    x, y = np.asarray(x), np.asarray(y)
    step = max(1, len(x) // n_marks)
    every = (min(len(x) - 1, (slot * step) // max(total, 1)), step) if len(x) > n_marks else 1   # stagger the markers
    return ax.plot(x, y, lw=1.4, ms=5, markevery=every, markerfacecolor="white", markeredgewidth=1.2, label=label, **look)[0]

def band(ax, x, lo, hi, look, label):
    ax.fill_between(x, lo, hi, color=look["color"], alpha=0.14, lw=0, label=label)
    ax.plot(x, lo, color=look["color"], lw=0.6, ls=look["ls"]); ax.plot(x, hi, color=look["color"], lw=0.6, ls=look["ls"])

def tidy(ax, nx=5, ny=5):
    ax.grid(True, alpha=0.25, lw=0.5)
    ax.xaxis.set_major_locator(MaxNLocator(nx)); ax.yaxis.set_major_locator(MaxNLocator(ny))

def finish(fig, ax_for_legend, name, ncol):
    h, l = ax_for_legend.get_legend_handles_labels()
    rows = math.ceil(len(l) / ncol)
    fig.legend(h, l, loc="upper center", ncol=ncol, frameon=False, bbox_to_anchor=(0.5, 1.0), columnspacing=1.4, handlelength=2.6)
    fig.tight_layout(rect=(0, 0, 1, 1 - (0.19 * rows + 0.06) / fig.get_figheight()), h_pad=1.0, w_pad=0.8)
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=600)
    plt.show()

# ---- Fig: NASA cells, actual vs predicted SOH (charging inputs), one panel per test cell ----
PRED_A = load('A6_v3')[1]
shown = ["Tuned XGBoost", "Random Forest", "CNN-LSTM"]
fig, axes = plt.subplots(2, 2, figsize=(PAGE_W, 4.3))
for ax, name in zip(axes.ravel(), BATTERIES):
    meta = PRED_A[("Charging", shown[0], name)]["meta"]
    ax.plot(meta["k"], meta["SOH"] * 100, label="Actual", **ACTUAL)
    for s, m in enumerate(shown):
        draw(ax, meta["k"], PRED_A[("Charging", m, name)]["mean"][:, 0] * 100, MODEL_LOOK[m], m, slot=s, total=len(shown))
    ax.set_title(f"Test cell {name}"); tidy(ax)
for ax in axes[1]: ax.set_xlabel("Discharge cycle")
for ax in axes[:, 0]: ax.set_ylabel("SOH (%)")
finish(fig, axes[0, 0], "fig_nasa_predictions", 4)

# ---- Fig: NASA cells, RUL with the calibrated 90% band ----
STREAM_B = load('B2_v4')[2]
fig, axes = plt.subplots(2, 2, figsize=(PAGE_W, 4.3))
for ax, name in zip(axes.ravel(), BATTERIES):
    m = STREAM_B[name]; t = "Tuned XGBoost (frozen)"
    ax.plot(m["k"], m["RUL"], label="Actual", **ACTUAL)
    band(ax, m["k"], m[t + " | low"], m[t + " | high"], LOOKS[0], "90% band (Tuned XGBoost)")
    draw(ax, m["k"], m[t], LOOKS[0], "Tuned XGBoost", slot=0, total=2)
    draw(ax, m["k"], m["Self-updating trend"], LOOKS[1], "Self-updating trend", slot=1, total=2)
    ax.set_title(f"Test cell {name}"); ax.set_ylim(bottom=0); tidy(ax)
for ax in axes[1]: ax.set_xlabel("Discharge cycle")
for ax in axes[:, 0]: ax.set_ylabel("RUL (cycles)")
finish(fig, axes[0, 0], "fig_nasa_rul_band", 2)

# ---- Fig: LFP cells, three example cells (shortest-lived, median, longest-lived) ----
life = matr.groupby("Battery", sort=False)["EOL_k"].first().values
picks = [int(np.argmin(life)), int(np.argsort(life)[len(life) // 2]), int(np.argmax(life))]
kinds = ["Shortest-lived", "Median", "Longest-lived"]
shown = ["Tuned XGBoost", "LSTM", "CNN-LSTM"]
fig, axes = plt.subplots(2, 3, figsize=(PAGE_W, 4.0), sharex="col")
for col, g in enumerate(picks):
    sel = G2 == g
    for row, (j, scale) in enumerate([(0, 100), (1, 1)]):
        ax = axes[row, col]
        ax.plot(K2[sel], Y2[sel, j] * scale, label="Actual", **ACTUAL)
        for s, mname in enumerate(shown):
            draw(ax, K2[sel], np.clip(PRED2[mname][sel, j], 0, None) * scale, MODEL_LOOK[mname], mname, n_marks=5, slot=s, total=len(shown))
        tidy(ax, nx=4)
    axes[0, col].set_title(f"{kinds[col]} cell\n({life[g]} cycles)"); axes[1, col].set_xlabel("Cycle")
axes[0, 0].set_ylabel("SOH (%)"); axes[1, 0].set_ylabel("RUL (cycles)")
finish(fig, axes[0, 0], "fig_lfp_examples", 4)

# ---- Fig: LFP cells, RUL with the calibrated 90% band ----
fig, axes = plt.subplots(1, 3, figsize=(PAGE_W, 2.5))
for col, (ax, g) in enumerate(zip(axes, picks)):
    sel = G2 == g
    ax.plot(K2[sel], Y2[sel, 1], label="Actual", **ACTUAL)
    band(ax, K2[sel], np.clip(E["xgb"][sel] - E["xgb_half"][sel], 0, None), E["xgb"][sel] + E["xgb_half"][sel], LOOKS[0], "90% band (Tuned XGBoost)")
    draw(ax, K2[sel], E["xgb"][sel], LOOKS[0], "Tuned XGBoost", n_marks=5, slot=0, total=2)
    draw(ax, K2[sel], E["twin"][sel], LOOKS[1], "With EOL filter", n_marks=5, slot=1, total=2)
    ax.set_title(f"{kinds[col]} cell\n({life[g]} cycles)"); ax.set_xlabel("Cycle"); ax.set_ylim(bottom=0); tidy(ax, nx=4)
axes[0].set_ylabel("RUL (cycles)")
finish(fig, axes[0], "fig_lfp_rul_band", 4)

# ---- Fig: SOC on two NASA cells (one late discharge, and the error over life); the table in the paper covers all four ----
soc_j = load('F1_v2')
err_j = lambda d, m: 100 * np.sqrt(((d[m] - d["SOC"]) ** 2).mean())
SOC_CELLS = [BATTERIES[0], BATTERIES[-1]]
fig, axes = plt.subplots(2, 2, figsize=(PAGE_W, 4.6))
for col, name in enumerate(SOC_CELLS):
    b = soc_j[soc_j["Battery"] == name]
    late = b[b["k"] == int(0.9 * b["k"].max())]
    axes[0, col].plot(late["Time_s"] / 60, late["SOC"] * 100, label="Reference", **ACTUAL)
    for s, m in enumerate(SOC_METHODS):
        draw(axes[0, col], late["Time_s"] / 60, late[m] * 100, LOOKS[s], m, slot=s, total=len(SOC_METHODS))
        per = b.groupby("k").apply(lambda d: err_j(d, m))
        draw(axes[1, col], per.index, per.values, LOOKS[s], m, slot=s, total=len(SOC_METHODS))
    axes[0, col].set_title(f"{name}, discharge cycle {int(late['k'].iloc[0])}"); axes[0, col].set_xlabel("Time in discharge (min)")
    axes[1, col].set_title(f"{name}, every discharge"); axes[1, col].set_xlabel("Discharge cycle"); tidy(axes[0, col]); tidy(axes[1, col])
axes[0, 0].set_ylabel("SOC (%)"); axes[1, 0].set_ylabel("SOC RMSE (points)")
finish(fig, axes[0, 0], "fig_soc", 2)

# ---- Fig: disturbed readings, the five models (needs Part I) ----
if "FAULT" in globals():
    fig, axes = plt.subplots(1, 3, figsize=(PAGE_W, 2.6), sharey=True)
    for ax, (fname, _, levels) in zip(axes, FAULTS):
        for mname in MODELS2:
            d = FAULT[(FAULT["Model"] == mname) & FAULT["Fault"].isin([fname, "Clean"])].sort_values("Level")
            draw(ax, 100 * d["Level"], d["RUL_RMSE"], MODEL_LOOK[mname], mname)
        ax.set_title(fname); ax.set_xlabel("Disturbance (%)"); tidy(ax)
    axes[0].set_ylabel("RUL RMSE (cycles)")
    finish(fig, axes[0], "fig_sensor_faults", 5)

# ---- Fig: disturbed readings, the Tuned XGBoost variants (needs Part I5) ----
if "ROBUST" in globals():
    fig, axes = plt.subplots(1, 3, figsize=(PAGE_W, 2.8), sharey=True)
    for ax, (fname, _, levels) in zip(axes, FAULTS):
        for s, v in enumerate(SUMMARIES):
            d = ROBUST[(ROBUST["Variant"] == v) & ROBUST["Fault"].isin([fname, "Clean"])].sort_values("Level")
            draw(ax, 100 * d["Level"], d["RUL_RMSE"], LOOKS[s], v)
        d = FAULT[(FAULT["Model"] == "LSTM") & FAULT["Fault"].isin([fname, "Clean"])].sort_values("Level")
        draw(ax, 100 * d["Level"], d["RUL_RMSE"], LOOKS[4], "LSTM")
        ax.set_title(fname); ax.set_xlabel("Disturbance (%)"); tidy(ax)
    axes[0].set_ylabel("RUL RMSE (cycles)")
    finish(fig, axes[0], "fig_robust_xgboost", 3)

# ---- Fig: input importance (needs Part I4) ----
if "IMP" in globals():
    NICE = {"T_1C_min": "Final 1C step: time", "Q_1C_Ah": "Final 1C step: charge", "T_CV_min": "Final CV step: time",
            "IR_ohm": "Internal resistance", "Chargetime_min": "Total charging time"}
    m = IMP[IMP["Group"] == "Input"].groupby("Name", sort=False)[["SOH", "RUL"]].agg(["mean", "std"])
    names = [NICE.get(n, n) for n in m.index]
    fig, axes = plt.subplots(1, 2, figsize=(PAGE_W, 2.3), sharey=True)
    for ax, target, unit, look, hatch in zip(axes, ["SOH", "RUL"], ["% points", "cycles"], LOOKS[:2], ["", "///"]):
        ax.barh(names, m[(target, "mean")], xerr=m[(target, "std")], height=0.6, color=look["color"], alpha=0.85, hatch=hatch,
                edgecolor="white", error_kw=dict(ecolor="black", lw=0.9, capsize=2.5))
        ax.set_xlabel(f"Rise in {target} RMSE ({unit})"); ax.grid(True, axis="x", alpha=0.25, lw=0.5); ax.xaxis.set_major_locator(MaxNLocator(5))
    axes[0].invert_yaxis()
    fig.tight_layout(w_pad=1.0)
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"fig_importance.{ext}"), dpi=600)
    plt.show()
plt.rcdefaults()
print("Saved:", sorted(os.listdir(FIG_DIR)))

# Part K — Additional analyses
- **K1.** Software versions and the processor used for the timing measurements.
- **K2.** Samples in one raw charging step compared with the few numbers the twin sends (set `COUNT_LFP = False` to skip the LFP count, which downloads one 2–3 GB batch file).
- **K3.** Effect sizes: median paired difference in per-cell RMSE with 95% bootstrap intervals.
- **K4.** LFP cells evaluated with folds grouped by charging policy, so every test cell was charged under a policy unseen in training. Retrains the five models: about 15–30 minutes on a T4 GPU (saved as `K16_v1`).

Needs Parts A, C and D to have run in the same session.

In [ ]:
import platform, sys, sklearn, xgboost, time
from scipy.io import loadmat
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
K_DIR = os.path.join(OUT, "paper_tables"); os.makedirs(K_DIR, exist_ok=True)
print("=" * 80)

# ---------------- K1: software and hardware ----------------
cpu = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), platform.processor())
ncpu = os.cpu_count()
ram = int(next(l.split()[1] for l in open("/proc/meminfo") if l.startswith("MemTotal"))) / 1024 ** 2
print("K1  Python", platform.python_version(), "| scikit-learn", sklearn.__version__, "| XGBoost", getattr(xgboost, "__version__", "?"),
      "| TensorFlow", getattr(tf, "__version__", "?"))
print(f"K1  Processor: {cpu}, {ncpu} logical cores, {ram:.1f} GB RAM, {platform.platform()}")
print("K1  Timing protocol in Part I1: 1 warm-up call, then 100 timed single-window predictions (median reported);",
      "the 1000-prediction column is one batched call over the whole test fold, scaled to 1000 windows.")
print("=" * 80)

# ---------------- K2: samples per charging step (raw data versus features) ----------------
nasa_n = []
for name in BATTERIES:
    cyc = loadmat(os.path.join(battery_path, name + ".mat"))[name][0, 0]["cycle"]
    for j in range(cyc.shape[1]):
        c = cyc[0, j]
        if str(c["type"][0]) == "charge":
            n = np.asarray(c["data"][0, 0]["Time"]).size
            if n >= 50:
                nasa_n.append(n)
nasa_med = float(np.median(nasa_n))
print(f"K2  NASA: median {nasa_med:.0f} samples per charging step (from {len(nasa_n)} charges); "
      f"raw {nasa_med * 12:.0f} bytes vs 16 bytes of features -> reduction factor about {nasa_med * 12 / 16:.0f}x")

COUNT_LFP = True          # set to False to skip (it downloads one LFP batch file, about 2-3 GB, if it is not already there)
if COUNT_LFP:
    import h5py
    url, fname = FILES["b2"]
    path = os.path.join(RAW, fname)
    if not (os.path.exists(path) and h5py.is_hdf5(path)):
        print("K2  downloading", fname, "to count LFP samples (a few minutes)...")
        download(url, path)
    lfp_n = []
    with h5py.File(path, "r") as f:
        batch = f["batch"]
        for i in range(batch["cycles"].shape[0]):
            cycrefs = f[batch["cycles"][i, 0]]
            Irefs, Qrefs = cycrefs["I"][:, 0], cycrefs["Qc"][:, 0]
            for j in range(1, len(Irefs), 50):                          # every 50th cycle of each cell
                I = np.asarray(f[Irefs[j]][()]).flatten(); Qc = np.asarray(f[Qrefs[j]][()]).flatten()
                if len(Qc) > 10 and Qc.max() > 0.5:
                    lfp_n.append(int(np.argmax(Qc)) + 1)                    # samples up to the end of charging
    lfp_med = float(np.median(lfp_n))
    print(f"K2  LFP: median {lfp_med:.0f} samples per charging step (from {len(lfp_n)} charges in {fname}); "
          f"raw {lfp_med * 12:.0f} bytes vs 20 bytes of features -> reduction factor about {lfp_med * 12 / 20:.0f}x")
print("=" * 80)

# ---------------- K3: effect size of the Wilcoxon comparisons ----------------
rng = np.random.default_rng(0)
cells_ = np.unique(G2)
def per_cell(mname, j, mult):
    p = PRED2[mname].copy(); p[:, 1] = np.clip(p[:, 1], 0, None)
    keep = np.ones(len(G2), bool) if j == 0 else PRE2
    return mult * np.array([np.sqrt(((Y2[(G2 == g) & keep, j] - p[(G2 == g) & keep, j]) ** 2).mean()) for g in cells_])
rows = []
for target, j, mult, unit in [("SOH", 0, 100, "%"), ("RUL", 1, 1, "cycles")]:
    ref = per_cell("Tuned XGBoost", j, mult)
    for m in [x for x in MODELS2 if x != "Tuned XGBoost"]:
        d = per_cell(m, j, mult) - ref                                   # positive = Tuned XGBoost better
        boots = [np.median(rng.choice(d, len(d), replace=True)) for _ in range(5000)]
        lo, hi = np.percentile(boots, [2.5, 97.5])
        rows.append(dict(Target=f"{target} ({unit})", Model=m, Median_difference=np.median(d), CI_low=lo, CI_high=hi))
EFF = pd.DataFrame(rows)
EFF.to_csv(os.path.join(K_DIR, "lfp_effect_sizes.csv"), index=False)
print("K3  Median paired difference in per-cell RMSE (other model minus Tuned XGBoost), 95% bootstrap interval")
print(EFF.round(3).to_string(index=False))
print("=" * 80)

# ---------------- K4: LFP evaluation with folds grouped by charging policy ----------------
policy_of_cell = matr.groupby("Battery", sort=False)["Policy"].first().values
POL = pd.factorize(policy_of_cell)[0][G2]                                # policy id of every window
print(f"K4  {len(np.unique(POL))} charging policies over {len(CELL_NAMES2)} cells")
if have('K16_v1'):
    PREDP = load('K16_v1')
else:
    PREDP = {m: np.full(Y2.shape, np.nan) for m in MODELS2}
    for fold, (tr, te) in enumerate(GroupKFold(n_splits=5).split(SEQ2, groups=POL)):
        t0 = time.time()
        xs = StandardScaler().fit(SEQ2[tr].reshape(-1, nf)); ys = StandardScaler().fit(Y2[tr])
        Str, Ste = (xs.transform(SEQ2[i].reshape(-1, nf)).reshape(-1, SEQ_LEN, nf) for i in (tr, te))
        ytr = ys.transform(Y2[tr]); Wtr, Wte = summary2(Str), summary2(Ste)
        PREDP["Linear"][te] = ys.inverse_transform(Ridge(alpha=1.0).fit(Str[:, -1, :], ytr).predict(Ste[:, -1, :]))
        rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(Wtr, ytr)
        PREDP["Random Forest"][te] = ys.inverse_transform(rf.predict(Wte))
        out = []
        for j in range(2):                                               # tuning grouped by policy as well
            search = RandomizedSearchCV(new_xgb(), XGB_SEARCH2, n_iter=XGB_TRIALS2, cv=GroupKFold(n_splits=3),
                                        scoring="neg_root_mean_squared_error", random_state=fold).fit(W_raw[tr], Y2[tr, j], groups=POL[tr])
            out.append(search.predict(W_raw[te]))
        PREDP["Tuned XGBoost"][te] = np.column_stack(out)
        set_seed(0)
        val_pol = np.random.choice(np.unique(POL[tr]), max(1, len(np.unique(POL[tr])) // 7), replace=False)
        va = np.isin(POL[tr], val_pol)
        for kind in ["LSTM", "CNN-LSTM"]:
            runs = []
            for seed in range(DEEP_SEEDS):
                set_seed(seed)
                m = fit_fast(build_model(nf, kind), Str[~va], ytr[~va], Str[va], ytr[va])
                runs.append(ys.inverse_transform(m.predict(Ste, verbose=0, batch_size=1024)))
            PREDP[kind][te] = np.mean(runs, axis=0)
        print(f"K4  fold {fold + 1}/5 done in {time.time() - t0:.0f} s")
    save('K16_v1', PREDP)

foldsP = [te for _, te in GroupKFold(n_splits=5).split(SEQ2, groups=POL)]
def fold_metrics(P, folds):
    rows = []
    for te in folds:
        pre = PRE2[te]; r = np.clip(P[te, 1], 0, None)
        rows.append({"SOH RMSE (%)": 100 * np.sqrt(mean_squared_error(Y2[te, 0], P[te, 0])),
                     "RUL RMSE (cycles)": np.sqrt(mean_squared_error(Y2[te, 1][pre], r[pre])),
                     "RUL MAE (cycles)": mean_absolute_error(Y2[te, 1][pre], r[pre]),
                     "RUL R2": r2_score(Y2[te, 1][pre], r[pre])})
    return pd.DataFrame(rows)
fmtk = lambda f: pd.Series({c: f"{f[c].mean():.{2 if 'SOH' in c or 'R2' in c else 0}f} ± {f[c].std(ddof=1):.{2 if 'SOH' in c or 'R2' in c else 0}f}" for c in f.columns})
tP = pd.DataFrame({m: fmtk(fold_metrics(PREDP[m], foldsP)) for m in MODELS2}).T; tP.index.name = "Model"
save_table("lfp_policy_grouped", tP, "LFP cells: SOH and RUL by model, 5-fold cross-validation grouped by charging policy (mean ± s.d. over folds).")

# Part L — NASA cells: single fixed split and charge and energy balance checks
- **L1. Single fixed split.** LSTM and CNN-LSTM trained on B0005, B0006 and B0007 and tested on B0018 alone, with the eight discharge summary features (Section "NASA Cells: A Single Fixed Split" of the paper). This experiment has its own labels: SOH relative to the cell's initial capacity, end of life at SOH ≤ 70% (or the last recorded cycle), and the record index of the NASA files as the cycle count, so RUL is in record-index units. Each model is the mean of five networks. GPU training is not bit-for-bit repeatable, so a rerun can differ slightly in the last digits.
- **L2. Charge and energy balance checks** on B0005 (Section "Charge and Energy Balance Checks"): Coulombic efficiency with a Gaussian fit and sample means, a third-degree polynomial correction of the cumulative energy imbalance, and L-BFGS-B correction coefficients for charge counting.

Needs the setup cell of Part A and the helper `save_table` of Part H.

## L1. Single fixed split

In [ ]:
# ---------------- L1: single fixed split (train B0005-B0007, test B0018), discharge summary features ----------------
import random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

L_FEATURES = ["Cycle", "Voltage_mean", "Voltage_min", "Voltage_max", "Current_mean",
              "Temperature_mean", "Temperature_max", "Discharge_time_s"]
L_TRAIN, L_TEST = ["B0005", "B0006", "B0007"], "B0018"
L_SEQ, L_SEEDS = 20, 5
L_MODELS = ["LSTM", "CNN-LSTM"]

def l_cell_table(name):
    """One row per discharge; Cycle is the record index of the NASA file (it also counts charge and impedance records)."""
    cyc = loadmat(os.path.join(battery_path, name + ".mat"))[name][0, 0]["cycle"]
    rows = []
    for i in range(cyc.shape[1]):
        c = cyc[0, i]
        if c["type"][0] != "discharge":
            continue
        d = c["data"][0, 0]
        v, cur, temp, t = (np.asarray(d[k]).flatten() for k in ("Voltage_measured", "Current_measured", "Temperature_measured", "Time"))
        rows.append(dict(Battery=name, Cycle=i + 1, Capacity_Ah=float(np.asarray(d["Capacity"]).flatten()[0]),
                         Voltage_mean=v.mean(), Voltage_min=v.min(), Voltage_max=v.max(), Current_mean=cur.mean(),
                         Temperature_mean=temp.mean(), Temperature_max=temp.max(), Discharge_time_s=t[-1] - t[0]))
    b = pd.DataFrame(rows)
    q0 = b["Capacity_Ah"].iloc[0]
    b["SOH"] = b["Capacity_Ah"] / q0                               # relative to the cell's initial capacity
    hit = b.index[b["Capacity_Ah"] <= 0.70 * q0]                   # end of life at SOH <= 70%, else the last recorded cycle
    eol = b.loc[hit[0], "Cycle"] if len(hit) else b["Cycle"].iloc[-1]
    b["RUL"] = np.maximum(eol - b["Cycle"], 0)
    return b

def l_build(kind):
    inp = keras.Input((L_SEQ, len(L_FEATURES)))
    x = inp
    if kind == "CNN-LSTM":                                         # two causal convolutions in front of the same LSTM
        x = layers.Conv1D(64, 3, padding="causal", activation="relu")(x)
        x = layers.Conv1D(64, 3, padding="causal", activation="relu")(x)
    x = layers.LSTM(64)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation="relu")(x)
    m = keras.Model(inp, layers.Dense(2)(x))
    m.compile(optimizer="adam", loss="mse")
    return m

def l_fit(m, X, y):
    m.fit(X, y, validation_split=0.2, epochs=100, batch_size=16, verbose=0,
          callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)])
    return m

if have('L1_v1'):
    L_ACTUAL, L_PRED = load('L1_v1')
else:
    tables = {n: l_cell_table(n) for n in L_TRAIN + [L_TEST]}
    train_rows = pd.concat([tables[n] for n in L_TRAIN])
    xs = StandardScaler().fit(train_rows[L_FEATURES].values)        # scalers fitted on the training cells only
    ys = StandardScaler().fit(train_rows[["SOH", "RUL"]].values)
    def l_windows(b):
        X, y = xs.transform(b[L_FEATURES].values), ys.transform(b[["SOH", "RUL"]].values)
        idx = range(len(X) - L_SEQ + 1)
        return np.array([X[i:i + L_SEQ] for i in idx]), np.array([y[i + L_SEQ - 1] for i in idx])
    Xtr, ytr = map(np.concatenate, zip(*[l_windows(tables[n]) for n in L_TRAIN]))
    Xte, yte = l_windows(tables[L_TEST])
    L_ACTUAL, L_PRED = ys.inverse_transform(yte), {}
    for kind in L_MODELS:
        runs = []
        for s in range(L_SEEDS):
            random.seed(s); np.random.seed(s); tf.random.set_seed(s)
            runs.append(l_fit(l_build(kind), Xtr, ytr).predict(Xte, verbose=0))
        L_PRED[kind] = ys.inverse_transform(np.mean(runs, axis=0))  # mean of the five networks
        print(f"{kind}: {L_SEEDS} networks trained")
    save('L1_v1', (L_ACTUAL, L_PRED))

def l_metrics(y, p, scale):
    nz = y != 0                                                    # MAPE only where the true value is not zero
    return {"RMSE": scale * np.sqrt(mean_squared_error(y, p)), "MAE": scale * mean_absolute_error(y, p),
            "MAPE (%)": 100 * np.mean(np.abs((p[nz] - y[nz]) / y[nz])), "R2": r2_score(y, p)}

rowsL = [{"Target": target, "Model": kind, **l_metrics(L_ACTUAL[:, j], L_PRED[kind][:, j], scale)}
         for j, (target, scale) in enumerate([("SOH (%)", 100.0), ("RUL (record-index units)", 1.0)]) for kind in L_MODELS]
tL = pd.DataFrame(rowsL).set_index(["Target", "Model"])
save_table("nasa_fixed_split", tL.round({"RMSE": 2, "MAE": 2, "MAPE (%)": 2, "R2": 3}),
           "NASA cells, single fixed split: trained on B0005-B0007, tested on B0018, discharge summary features (mean of five networks).")
for target, scale in [("SOH (%)", 100.0), ("RUL (record-index units)", 1.0)]:
    a, b = tL.loc[(target, "LSTM"), "RMSE"], tL.loc[(target, "CNN-LSTM"), "RMSE"]
    print(f"{target}: CNN-LSTM RMSE {100 * (a - b) / a:+.1f}% lower than LSTM")

## L2. Charge and energy balance checks

In [ ]:
# ---------------- L2: charge and energy balance checks on B0005 ----------------
from scipy.stats import norm
from scipy.optimize import minimize

cyc5 = loadmat(os.path.join(battery_path, "B0005.mat"))["B0005"][0, 0]["cycle"]
def l_arr(c, key): return np.asarray(c["data"][0, 0][key]).flatten()
def l_integral(y, t): return np.sum((y[:-1] + y[1:]) / 2 * np.diff(t)) / 3600     # trapezoidal rule, per hour

pairs = [(i, i + 1) for i in range(cyc5.shape[1] - 1)
         if cyc5[0, i]["type"][0] == "charge" and cyc5[0, i + 1]["type"][0] == "discharge"]
bal = []
for i, j in pairs:
    ch, dis = cyc5[0, i], cyc5[0, j]
    i_ch, t_ch = l_arr(ch, "Current_charge"), l_arr(ch, "Time")
    q_ch = np.sum(np.abs((i_ch[:-1] + i_ch[1:]) / 2) * np.diff(t_ch)) / 3600
    q_dis = float(l_arr(dis, "Capacity")[0])
    e_ch = abs(l_integral(l_arr(ch, "Voltage_measured") * l_arr(ch, "Current_measured"), l_arr(ch, "Time")))
    e_dis = abs(l_integral(l_arr(dis, "Voltage_measured") * l_arr(dis, "Current_measured"), l_arr(dis, "Time")))
    bal.append(dict(Charge_record=i + 1, Q_charge_Ah=q_ch, Q_discharge_Ah=q_dis, CE=q_dis / q_ch if q_ch > 0 else np.nan,
                    Energy_imbalance_Wh=e_ch - e_dis))
bal = pd.DataFrame(bal)

# Coulombic efficiency: charges outside 1.0-2.5 Ah are incomplete or irregular
valid = bal[(bal["Q_charge_Ah"] > 1.0) & (bal["Q_charge_Ah"] < 2.5)].copy()
ce = valid["CE"].dropna().values
ce_mean, ce_std = norm.fit(ce)
means5 = np.array([ce[i:i + 5].mean() for i in range(0, len(ce) - 4, 5)])
print(f"Coulombic efficiency: {len(ce)} valid pairs, Gaussian mean {ce_mean:.4f}, standard deviation {ce_std:.4f}")
print(f"Sample means of 5: {len(means5)} means, mean {means5.mean():.4f}, standard deviation {means5.std():.4f}")

# Energy balance: third-degree polynomial fitted to the cumulative imbalance over the valid pairs
k = np.arange(len(valid))
cum = valid["Energy_imbalance_Wh"].cumsum().values
poly = np.poly1d(np.polyfit(k, cum, 3))
corrected = cum - poly(k)
print(f"Energy: cumulative imbalance reaches {cum[-1]:.2f} Wh; after the polynomial correction the largest residual is {np.abs(corrected).max():.3f} Wh")

# Correction coefficients for charge counting (L-BFGS-B); the reference SOC is built from the same records,
# so this demonstrates the procedure and gives no accuracy figure
RATED_AH = 2.0
steps = []
for i, j in pairs:
    for idx, mode in [(i, "charge"), (j, "discharge")]:
        cur, t = l_arr(cyc5[0, idx], "Current_measured"), l_arr(cyc5[0, idx], "Time")
        dq = np.abs((cur[:-1] + cur[1:]) / 2) * np.diff(t) / 3600
        if dq.sum() <= 0:
            continue
        ref = np.cumsum(dq) / dq.sum() if mode == "charge" else 1 - np.cumsum(dq) / dq.sum()
        steps.append((mode, dq, ref))
def soc_error(theta):
    nc, dc = theta
    err = [ref - (np.cumsum(nc * dq) / RATED_AH if mode == "charge" else 1 - np.cumsum(dc * dq) / RATED_AH)
           for mode, dq, ref in steps]
    return np.mean(np.square(np.concatenate(err)))
fit = minimize(soc_error, x0=[1.0, 1.0], method="L-BFGS-B", bounds=[(0.0, 2.0), (0.0, 2.0)])
print(f"L-BFGS-B: converged {fit.success}, charge coefficient {fit.x[0]:.4f}, discharge coefficient {fit.x[1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(ce, bins=12, density=True, color="#9db7d5", edgecolor="white", label="Valid CE values")
xs_ = np.linspace(ce.min(), ce.max(), 300)
axes[0].plot(xs_, norm.pdf(xs_, ce_mean, ce_std), color="#2a6fb0", ls="-", label="Gaussian fit")
axes[0].axvline(ce_mean, color="#a8483c", ls="--", label=f"Mean {ce_mean:.4f}")
axes[0].set_xlabel("Coulombic efficiency"); axes[0].set_ylabel("Density"); axes[0].legend(fontsize=8)
axes[1].plot(k, cum, color="#2a6fb0", marker="o", ms=3, ls="-", label="Cumulative imbalance")
axes[1].plot(k, poly(k), color="#c9742a", ls="--", label="Third-degree fit")
axes[1].plot(k, corrected, color="#2f8f6b", marker="^", ms=3, ls="-.", label="Corrected")
axes[1].set_xlabel("Valid charge-discharge pair"); axes[1].set_ylabel("Energy (Wh)"); axes[1].legend(fontsize=8)
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "nasa_balance_checks.png"), dpi=300); plt.show()